# Retrieval-Augmented Generation (RAG) for InvestIQ

---

# 0. Import required libraries
---

In [ ]:
!pip install -q -U langchain-text-splitters sentence-transformers huggingface_hub chromadb torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7

In [ ]:
import torch
import transformers
import peft
import chromadb
import sentence_transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("ChromaDB:", chromadb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cpu
Transformers: 5.16.1
PEFT: 0.20.0
ChromaDB: 1.5.9
Sentence Transformers: 6.1.0
CUDA available: False


---
# 1. Load the RAG Source Documents for LMT and WBD

For this project, I will use the 10-K filings of two companies:

- Lockheed Martin {{'SimFinId': 103644}, {'Ticker': 'LMT'}}
- Warner Bros. Discovery {{'SimFinId': 120376}, {'Ticker': 'WBD'}}

These documents will be used as the source material for the RAG pipeline.

In [ ]:
LOCKHEED_PATH = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/LMT/10-K_2026-01-29_0001628280-26-004195.txt"

WBD_PATH = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/WBD/10-K_2026-02-27_0001437107-26-000020.txt"

In [ ]:
# Read both 10-K documents

with open(LOCKHEED_PATH, "r", encoding="utf-8") as file:
    lockheed_text = file.read()

with open(WBD_PATH, "r", encoding="utf-8") as file:
    wbd_text = file.read()

print("Lockheed Martin characters:", len(lockheed_text))
print("Warner Bros. Discovery characters:", len(wbd_text))

print("\nLockheed Martin preview:")
print(lockheed_text[:500])

print("\nWarner Bros. Discovery preview:")
print(wbd_text[:500])

Lockheed Martin characters: 493420
Warner Bros. Discovery characters: 606435

Lockheed Martin preview:
lmt-20251231
false
FY
0000936468
2025
P5Y
P2Y
P3Y
http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://fasb.org/us-gaap/2025#OtherAssetsCurrent http://fasb.org/us-gaap/2025#OtherAssetsNoncurrent
http://fasb.org/us-gaap/2025#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2025#OtherLiabilitiesNoncurrent
http://xbrl.sec.gov/country/2025#US
http://xbrl.sec.gov/country/2025#US
http://xbrl.sec.gov/country/2025#US
P8Y
P8Y
P1Y
http://fasb.org

Warner Bros. Discovery preview:
wbd-20251231
0001437107
2025
FY
false
P1Y
P3Y0D
P3Y0D
5
7
37.02
7
4
http://fasb.org/us-gaap/2025#CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization
http://fasb.org/us-gaap/2025#CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization
http://fasb.org/us-gaap/2025#CostOfGoodsAndServiceExcludingDepreciationDepletionAndAmortization
http://

## 1.1 Inspect the Source Document Structure

The source files are readable, but their beginning contains `SEC/XBRL metadata` and `accounting tags`.

Before building the **RAG** pipeline, I will inspect the documents to determine whether they also contain the narrative text of the $10-K$ filings.

This is important because **RAG** should retrieve meaningful financial and business information rather than primarily retrieving technical `XBRL metadata`.

In [ ]:
# Look for common narrative sections in each document

keywords = [
    "Business",
    "Risk Factors",
    "Management's Discussion and Analysis",
    "Management’s Discussion and Analysis",
    "Financial Statements",
]

for name, text in [
    ("Lockheed Martin", lockheed_text),
    ("Warner Bros. Discovery", wbd_text)
]:
    print(f"\n===== {name} =====")

    for keyword in keywords:
        position = text.find(keyword)

        if position != -1:
            print(f"{keyword}: found at character {position}")
        else:
            print(f"{keyword}: not found")


===== Lockheed Martin =====
Business: found at character 62548
Risk Factors: found at character 62568
Management's Discussion and Analysis: not found
Management’s Discussion and Analysis: found at character 62936
Financial Statements: found at character 63104

===== Warner Bros. Discovery =====
Business: found at character 29395
Risk Factors: found at character 74508
Management's Discussion and Analysis: not found
Management’s Discussion and Analysis: found at character 74843
Financial Statements: found at character 75013


## 1.2 Inspect a Narrative Section

The source files contain both structured `SEC/XBRL` information and `10-K` narrative content.

I will now inspect a small section of the narrative text to determine how clean and readable the source material is before designing the chunking step.

In [ ]:
# Inspect text around the Business section
for name, text in [
    ("Lockheed Martin", lockheed_text),
    ("Warner Bros. Discovery", wbd_text)
]:
  position = text.find("Business")

  print(f"\n==== {name} ====")
  print(text[position:position + 3000])


==== Lockheed Martin ====
Business
3
ITEM 1A.
Risk Factors
10
ITEM 1B.
Unresolved Staff Comments
23
ITEM 1C.
Cybersecurity
23
ITEM 2.
Properties
24
ITEM 3.
Legal Proceedings
25
ITEM 4.
Mine Safety Disclosures
25
Information about our Executive Officers
25
PART II
ITEM 5.
Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities
27
ITEM 6.
[Reserved]
28
ITEM 7.
Management’s Discussion and Analysis of Financial Condition and Results of Operations
29
ITEM 7A.
Quantitative and Qualitative Disclosures About Market Risk
49
ITEM 8.
Financial Statements and Supplementary Data
50
ITEM 9.
Changes in and Disagreements with Accountants on Accounting and Financial Disclosure
93
ITEM 9A.
Controls and Procedures
93
ITEM 9B.
Other Information
95
ITEM 9C.
Disclosure Regarding Foreign Jurisdictions that Prevent Inspections
95
PART III
ITEM 10.
Directors, Executive Officers and Corporate Governance
95
ITEM 11.
Executive Compensation
95
ITEM 12.
Security

## 1.3 Locate the Main 10-K Narrative

The source files contain both `XBRL metadata` and `narrative filing content`.

The **Lockheed Martin file** clearly contains the 10-K table of contents and narrative text. For **Warner Bros. Discovery**, the first occurrence of *"Business"* was inside XBRL metadata.

I will therefore locate the main `ITEM 1.` section before deciding how to clean the documents.

In [ ]:
# Inspect the exact characters around "ITEM 1. Business"

for name, text in [
    ("Lockheed Martin", lockheed_text),
    ("Warner Bros. Discovery", wbd_text)
]:
    print(f"\n========== {name} ==========")

    position = text.find("ITEM")

    print("First 'ITEM' found at:", position)

    if position != -1:
        sample = text[position:position + 30]

        print("\nVisible text:")
        print(sample)

        print("\nCharacter representations:")
        print(repr(sample))


========== Lockheed Martin ==========
First 'ITEM' found at: 62540

Visible text:
ITEM 1.
Business
3
ITEM 1A.
Ri

Character representations:
'ITEM\xa01.\nBusiness\n3\nITEM\xa01A.\nRi'

========== Warner Bros. Discovery ==========
First 'ITEM' found at: 74479

Visible text:
ITEM 1. Business.
6
ITEM 1A. R

Character representations:
'ITEM\xa01. Business.\n6\nITEM\xa01A. R'


In [ ]:
# Normalize non-breaking spaces

lockheed_clean = lockheed_text.replace("\xa0", " ")
wbd_clean = wbd_text.replace("\xa0", " ")

print("Lockheed Martin:")
print("Before:", len(lockheed_text))
print("After: ", len(lockheed_clean))

print("\nWarner Bros. Discovery:")
print("Before:", len(wbd_text))
print("After: ", len(wbd_clean))

Lockheed Martin:
Before: 493420
After:  493420

Warner Bros. Discovery:
Before: 606435
After:  606435


In [ ]:
# Verify that the non-breaking spaces were replaced

print("Lockheed Martin:")
print("Raw search:  ", lockheed_text.find("ITEM 1."))
print("Clean search:", lockheed_clean.find("ITEM 1."))

print("\nWarner Bros. Discovery:")
print("Raw search:  ", wbd_text.find("ITEM 1."))
print("Clean search:", wbd_clean.find("ITEM 1."))

Lockheed Martin:
Raw search:   -1
Clean search: 62540

Warner Bros. Discovery:
Raw search:   -1
Clean search: 74479


In [ ]:
# Find occurrences of "ITEM 1." in each document
for name, text in [
    ("Lockheed Martin", lockheed_clean),
    ("Warner Bros. Discovery", wbd_clean)
]:
  print(f"\n==== {name} ====")

  start = 0
  count = 0

  while True:
    position = text.find("ITEM 1.", start)

    if position == -1:
      break

    print(f"\n--- Occurrence {count + 1} at character {position} ---")
    print(text[position:position + 1000])

    start = position + 1
    count += 1

### 1.3.1 Locate the Main 10-K Section Boundaries

The cleaned documents contain multiple sections from the 10-K filing.

Before extracting the narrative content, I will locate the main section headings and their positions in each document.

These positions will help us identify the boundaries between sections such as Business, Risk Factors, Management's Discussion and Analysis, and Financial Statements.

In [ ]:
section_headers = [
    "ITEM 1. Business",
    "ITEM 1A. Risk Factors",
    "ITEM 7. Management’s Discussion and Analysis",
    "ITEM 8. Financial Statements",
]

for name, text in [
    ("Lockheed Martin", lockheed_clean),
    ("Warner Bros. Discovery", wbd_clean)
]:
    print(f"\n========== {name} ==========")

    for header in section_headers:
        print(f"\n{header}")

        start = 0

        while True:
            position = text.find(header, start)

            if position == -1:
                break

            print("  ", position)

            start = position + 1

### 1.3.2 Inspect Lockheed Martin Section Heading Formatting

The Lockheed Martin filing uses a different formatting pattern from Warner Bros. Discovery.

For example, the Business heading is split across two lines:

`ITEM 1.`

`Business`

I will inspect the formatting of several Lockheed Martin section headings before creating the extraction logic.

In [ ]:
lockheed_headers = [
    "ITEM 1.",
    "ITEM 1A.",
    "ITEM 7.",
    "ITEM 8."
]

for header in lockheed_headers:
    print(f"\n===== {header} =====")

    start = 0
    count = 0

    while True:
        position = lockheed_clean.find(header, start)

        if position == -1:
            break

        print(f"\nOccurrence {count + 1} at character {position}:")
        print(repr(lockheed_clean[position:position + 80]))

        start = position + 1
        count += 1

---
# 2. Verify the Business Section Boundary

The actual `ITEM 1. Business` section should end when the actual `ITEM 1A. Risk Factors` section begins.

I will inspect the text immediately before and after this boundary for both companies to make sure the section extraction will preserve the correct narrative content.

In [ ]:
business_headers = {
    "Lockheed Martin": ("ITEM 1.\nBusiness", "ITEM 1A."),
    "Warner Bros. Discovery": ("ITEM 1. Business.", "ITEM 1A.")
}

for name, (business_header, risk_header) in business_headers.items():

    text = lockheed_clean if name == "Lockheed Martin" else wbd_clean

    business_positions = []
    start = 0

    while True:
        position = text.find(business_header, start)

        if position == -1:
            break

        business_positions.append(position)
        start = position + 1

    risk_positions = []
    start = 0

    while True:
        position = text.find(risk_header, start)

        if position == -1:
            break

        risk_positions.append(position)
        start = position + 1

    # Use the last occurrence of each heading,
    # which corresponds to the actual filing narrative.
    business_start = business_positions[-1]
    risk_start = risk_positions[-1]

    print(f"\n========== {name} ==========")
    print("Business starts at:", business_start)
    print("Risk Factors starts at:", risk_start)
    print("Business section length:", risk_start - business_start)

    print("\n--- End of Business section ---")
    print(text[risk_start - 500:risk_start + 300])

## 2.1 Define the RAG Section Boundaries

For the RAG baseline, I will extract four sections from each company's 10-K filing:

1. Business
2. Risk Factors
3. Management's Discussion and Analysis (MD&A)
4. Financial Statements and related notes

The section boundaries will be determined from the actual narrative occurrences of the 10-K headings rather than fixed character positions.

The complete Financial Statements section will be preserved, including its tables and notes.

In [ ]:
section_patterns = {
    "Business": [
        "ITEM 1.\nBusiness",
        "ITEM 1. Business"
    ],
    "Risk Factors": [
        "ITEM 1A.\nRisk Factors",
        "ITEM 1A.    Risk Factors",
        "ITEM 1A. Risk Factors"
    ],
    "MD&A": [
        "ITEM 7.\nManagement’s Discussion and Analysis",
        "ITEM 7. Management’s Discussion and Analysis"
    ],
    "Financial Statements": [
        "ITEM 8.\nFinancial Statements",
        "ITEM 8. Financial Statements"
    ]
}

print("Section patterns defined:")
for section, patterns in section_patterns.items():
    print(f"\n{section}:")
    for pattern in patterns:
        print(repr(pattern))

## 2.2 Locate the Actual 10-K Section Headings

The same section heading can appear in the Table of Contents and again in the actual filing narrative.

For each section, I will find all matching occurrences and use the last occurrence as the starting position of the actual narrative section.

In [ ]:
def find_actual_section_start(text, patterns):
    positions = []

    for pattern in patterns:
        start = 0

        while True:
            position = text.find(pattern, start)

            if position == -1:
                break

            positions.append(position)
            start = position + 1

    if not positions:
        return None

    return max(positions)


for name, text in [
    ("Lockheed Martin", lockheed_clean),
    ("Warner Bros. Discovery", wbd_clean)
]:
    print(f"\n========== {name} ==========")

    for section, patterns in section_patterns.items():
        position = find_actual_section_start(text, patterns)

        print(f"{section:<22} {position}")

---
# 3. Extract the Four RAG Sections

The actual starting positions of the four 10-K sections have been verified.

I will now extract the text belonging to each section:

- Business
- Risk Factors
- Management's Discussion and Analysis (MD&A)
- Financial Statements and related notes

The extracted sections will be stored together with their company and section metadata.

In [ ]:
def extract_sections(text, section_patterns):
    section_starts = {}

    # Find the actual starting position of each section
    for section, patterns in section_patterns.items():
        position = find_actual_section_start(text, patterns)

        if position is None:
            raise ValueError(f"Could not find section: {section}")

        section_starts[section] = position

    # Sort sections by their position in the document
    ordered_sections = sorted(
        section_starts.items(),
        key=lambda item: item[1]
    )

    extracted = {}

    # Extract each section up to the beginning of the next section
    for i, (section, start) in enumerate(ordered_sections):

        if i < len(ordered_sections) - 1:
            end = ordered_sections[i + 1][1]
            section_text = text[start:end]
        else:
            section_text = text[start:]

        extracted[section] = section_text.strip()

    return extracted


lockheed_sections = extract_sections(
    lockheed_clean,
    section_patterns
)

wbd_sections = extract_sections(
    wbd_clean,
    section_patterns
)

print("Lockheed Martin:")
for section, text in lockheed_sections.items():
    print(f"{section:<22} {len(text):>8} characters")

print("\nWarner Bros. Discovery:")
for section, text in wbd_sections.items():
    print(f"{section:<22} {len(text):>8} characters")

## 3.1 Validate the Extracted Sections

Before chunking and embedding the extracted sections, I will inspect their beginnings and endings.

This verifies that each section contains the intended 10-K content and that the section boundaries were applied correctly.

In [ ]:
for name, sections in [
    ("Lockheed Martin", lockheed_sections),
    ("Warner Bros. Discovery", wbd_sections)
]:
    print(f"\n\n========== {name} ==========")

    for section, text in sections.items():
        print(f"\n--- {section}: beginning ---")
        print(text[:300])

        print(f"\n--- {section}: ending ---")
        print(text[-300:])

### 3.1.1 Verify the RAG Section End Boundaries

The previous extraction showed that using the next selected RAG section as the boundary can accidentally include unrelated 10-K sections.

I will therefore verify the first actual section heading after each of the four RAG sections.

The intended boundaries are:

- Business → ITEM 1A. Risk Factors
- Risk Factors → ITEM 1B. Unresolved Staff Comments
- MD&A → ITEM 7A. Quantitative and Qualitative Disclosures About Market Risk
- Financial Statements → ITEM 9. Changes in and Disagreements With Accountants on Accounting and Financial Disclosure

In [ ]:
boundary_patterns = {
    "Business": [
        "ITEM 1A."
    ],
    "Risk Factors": [
        "ITEM 1B."
    ],
    "MD&A": [
        "ITEM 7A."
    ],
    "Financial Statements": [
        "ITEM 9.\nChanges in and Disagreements",
        "ITEM 9. Changes in and Disagreements"
    ]
}

for name, text in [
    ("Lockheed Martin", lockheed_clean),
    ("Warner Bros. Discovery", wbd_clean)
]:
    print(f"\n========== {name} ==========")

    for section, patterns in boundary_patterns.items():

        section_start = find_actual_section_start(
            text,
            section_patterns[section]
        )

        boundary_position = find_actual_section_start(
            text,
            patterns
        )

        print(f"\n{section}")
        print("Start:", section_start)
        print("End boundary:", boundary_position)

        if section_start is not None and boundary_position is not None:
            print("Section length:", boundary_position - section_start)

            print("\nBoundary text:")
            print(text[boundary_position:boundary_position + 250])

## 3.2 Extract the Final RAG Sections

The section boundaries have been verified for both 10-K documents.

I will now extract the four sections selected for the common RAG baseline:

1. Business
2. Risk Factors
3. Management's Discussion and Analysis (MD&A)
4. Financial Statements and related notes

Each section will end at the next relevant 10-K boundary so that unrelated sections are not included.

In [ ]:
def extract_rag_sections(text):
    sections = {}

    # Find the verified starting positions of the four RAG sections
    business_start = find_actual_section_start(
        text,
        section_patterns["Business"]
    )

    risk_start = find_actual_section_start(
        text,
        section_patterns["Risk Factors"]
    )

    mda_start = find_actual_section_start(
        text,
        section_patterns["MD&A"]
    )

    financial_start = find_actual_section_start(
        text,
        section_patterns["Financial Statements"]
    )

    # Find the boundaries where each selected section should end
    risk_boundary = find_actual_section_start(
        text,
        boundary_patterns["Risk Factors"]
    )

    mda_boundary = find_actual_section_start(
        text,
        boundary_patterns["MD&A"]
    )

    financial_boundary = find_actual_section_start(
        text,
        boundary_patterns["Financial Statements"]
    )

    # Business ends where Risk Factors begins
    sections["Business"] = text[
        business_start:risk_start
    ].strip()

    # Risk Factors ends at ITEM 1B
    sections["Risk Factors"] = text[
        risk_start:risk_boundary
    ].strip()

    # MD&A ends at ITEM 7A
    sections["MD&A"] = text[
        mda_start:mda_boundary
    ].strip()

    # Financial Statements ends at ITEM 9
    sections["Financial Statements"] = text[
        financial_start:financial_boundary
    ].strip()

    return sections


lockheed_rag_sections = extract_rag_sections(lockheed_clean)
wbd_rag_sections = extract_rag_sections(wbd_clean)


print("Lockheed Martin:")
for section, text in lockheed_rag_sections.items():
    print(f"{section:<22} {len(text):>8} characters")

print("\nWarner Bros. Discovery:")
for section, text in wbd_rag_sections.items():
    print(f"{section:<22} {len(text):>8} characters")

---
# 4. Validate the Final RAG Sections

The four RAG sections have now been extracted using the verified 10-K boundaries.

Before chunking the text, I will perform a final boundary check to make sure sections that were intentionally excluded are not present inside the extracted content.

In [ ]:
checks = {
    "Risk Factors": ["ITEM 1B.", "ITEM 1C."],
    "MD&A": ["ITEM 7A."],
    "Financial Statements": ["ITEM 9A.", "ITEM 9B.", "ITEM 9C."]
}

for name, sections in [
    ("Lockheed Martin", lockheed_rag_sections),
    ("Warner Bros. Discovery", wbd_rag_sections)
]:
    print(f"\n========== {name} ==========")

    for section, unwanted_headers in checks.items():

        text = sections[section]

        print(f"\n{section}")

        for header in unwanted_headers:
            found = header in text

            print(f"  {header:<10} {'FOUND' if found else 'not found'}")

---
# 5. Create the RAG Chunks

The four extracted 10-K sections are still too large to retrieve efficiently as complete sections.

I will divide each section into smaller chunks using recursive character-based splitting.

Baseline configuration:

- Chunk size: 3,000 characters
- Chunk overlap: 300 characters

The chunks will later be converted into embeddings and stored in the vector database.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Baseline chunking configuration
CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300

text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)

## 5.1 Split the Extracted Sections into Chunks

Each extracted section will now be divided into smaller chunks using the baseline text splitter.

The company and section information will be preserved for every chunk so that we can attach complete metadata before storing the chunks in the vector database.

In [ ]:
def create_chunks(company, sections):
    chunks = []

    for section, text in sections.items():
        section_chunks = text_splitter.split_text(text)

        for index, chunk in enumerate(section_chunks, start=1):
            chunks.append({
                "company": company,
                "section": section,
                "chunk_number": index,
                "text": chunk
            })

    return chunks

lockheed_chunks = create_chunks("Lockheed Martin", lockheed_rag_sections)

wbd_chunks = create_chunks("Warner Bros. Discovery", wbd_rag_sections)

print("Lockheed Martin chunks:", len(lockheed_chunks))
print("Warner Bros. Discovery chunks:", len(wbd_chunks))
print("Total chunks:", len(lockheed_chunks) + len(wbd_chunks))

### 5.1.2 Inspect the Generated Chunks

The two companies produced 322 chunks using the baseline chunking configuration.

Before creating embeddings, I will inspect representative chunks from different sections to verify that the chunks preserve meaningful context and do not contain obvious boundary problems.

In [ ]:
# Inspect representative chunks from both companies
for name, chunks in [
    ("Lockheed Martin", lockheed_chunks),
    ("Warner Bros. Discovery", wbd_chunks)
]:
    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")

    # Inspect the first chunk and a middle chunk
    sample_indexes = [0, len(chunks) // 2]

    for index in sample_indexes:
        chunk = chunks[index]

        print(f"\n--- Chunk {index + 1} ---")
        print("Section:", chunk["section"])
        print("Characters:", len(chunk["text"]))
        print("\nText:")
        print(chunk["text"][:1500])

---
# 6. Add Standardized Metadata

Each RAG chunk will be associated with standardized metadata identifying its company, filing, section, and source.

The same metadata structure will be used for both companies so that the resulting RAG data can later be combined with the other six companies in the project.

*Example metadata structure:*

| Field	|Lockheed example|
|-------- |--------------|
| company	| Lockheed Martin |
| ticker	| LMT |
| filing_type	| 10-K |
| filing_year	| 2025 |
| section	| MD&A |
| chunk_id	| LMT_MDA_075 |
| source | SEC EDGAR |

> *The `chunk_id` is generated automatically.*
---


In [ ]:
# Standard metadata for the two companies
company_metadata = {
    "Lockheed Martin": {
        "ticker": "LMT",
        "filing_type": "10-K",
        "filing_year": 2025,
        "source": "SEC EDGAR"
    },
    "Warner Bros. Discovery": {
        "ticker": "WBD",
        "filing_type": "10-K",
        "filing_year": 2025,
        "source": "SEC EDGAR"
    }
}

def add_metadata(chunks):
    final_chunks = []

    for chunk in chunks:
        company = chunk["company"]
        section = chunk["section"]
        chunk_number = chunk["chunk_number"]

        metadata = company_metadata[company]

        final_chunks.append({
            "text": chunk["text"],
            "metadata": {
                "company": company,
                "ticker": metadata["ticker"],
                "filing_type": metadata["filing_type"],
                "filing_year": metadata["filing_year"],
                "section": section,
                "chunk_id": f"{metadata['ticker']}_{section.replace(' ', '_')}_{chunk_number:03d}",
                "source": metadata["source"]
            }
        })

    return final_chunks

lockheed_final_chunks = add_metadata(lockheed_chunks)
wbd_final_chunks = add_metadata(wbd_chunks)

all_chunks = lockheed_final_chunks + wbd_final_chunks

print("Lockheed Martin chunks:", len(lockheed_final_chunks))
print("Warner Bros. Discovery chunks:", len(wbd_final_chunks))
print("Total chunks:", len(all_chunks))

## 6.1 Inspect a Final RAG Chunk Record

The chunk text and standardized metadata have now been combined into the structure that will later be stored in the vector database.

I will inspect one complete record to verify that the text and metadata are correctly associated.

In [ ]:
# Inspect one complete chunk record
sample_chunk = all_chunks[0]

print("Text:")
print(sample_chunk["text"][:1000])

print("\nMetadata:")

for key, value in sample_chunk["metadata"].items():
    print(f"{key}: {value}")

---
# 7. Validate RAG Chunk Metadata

Before creating embeddings, verify that every chunk contains the required metadata fields.

In [ ]:
required_metadata = [
    "company",
    "ticker",
    "filing_type",
    "filing_year",
    "section",
    "chunk_id",
    "source"
]

for index, chunk in enumerate(all_chunks):
    metadata = chunk["metadata"]

    missing = [key for key in required_metadata if key not in metadata]

    if missing:
        print(f"Chunk {index + 1} is missing: {missing}")

print("Metadata validation complete.")

---

# Checkpoint

An embedding is a numerical representation of text meaning that allows us to compare a user's question with document chunks by semantic similarity.

**The important distinction**:

> A **smaller cosine distance** means more similar.

> A **larger cosine similarity** means more similar.

For cosine similarity:

$$similarity=\frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\|\|\mathbf{B}\|}$$

---



# 8. Load the Embedding Model

The embedding model converts text into numerical vectors that represent semantic meaning.

For the RAG baseline, we use `all-MiniLM-L6-v2`.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    # Attempt to retrieve token
    hf_token = userdata.get('HugFace_TOKEN')

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    # If the token isn't found, smoothly continue without crashing
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Authentication successful.


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


## 8.1 Create a Test Embedding

Convert one RAG chunk into an embedding vector to verify that the embedding model is working correctly.

In [ ]:
sample_text = all_chunks[0]["text"]

sample_embedding = embedding_model.encode(sample_text)

print("Embedding type:", type(sample_embedding))
print("Embedding dimensions:", len(sample_embedding))
print("First 10 values:", sample_embedding[:10])

### 8.1.1 Test Semantic Similarity

Compare the embedding of a RAG document chunk with embeddings of related and unrelated questions using cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

related_question = "What does Lockheed Martin do?"
unrelated_question = "What are the symptoms of influenza?"

related_embedding = embedding_model.encode(related_question)
unrelated_embedding = embedding_model.encode(unrelated_question)

related_similarity = cosine_similarity(
    [sample_embedding],
    [related_embedding]
)[0][0]

unrelated_similarity = cosine_similarity(
    [sample_embedding],
    [unrelated_embedding]
)[0][0]

print("Related question similarity:", related_similarity)
print("Unrelated question similarity:", unrelated_similarity)

---
# 9. Create Embeddings for All RAG Chunks

Convert all 322 RAG chunks into embedding vectors while keeping the original chunk order.

In [ ]:
all_texts = [chunk["text"] for chunk in all_chunks]

all_embeddings = embedding_model.encode(all_texts, show_progress_bar=True)

print("Number of embeddings:", len(all_embeddings))
print("Embedding dimensions:", all_embeddings.shape[1])

## 9.1 Validate Chunk-Embedding Alignment

Verify that every RAG chunk has exactly one corresponding embedding vector.

In [ ]:
print("Number of chunks:", len(all_chunks))
print("Number of embeddings:", len(all_embeddings))

assert len(all_chunks) == len(all_embeddings)

print("Chunk–embedding alignment verified.")

---
# 10. Set Up ChromaDB

ChromaDB will store the RAG embeddings, document text, and metadata so that relevant chunks can be retrieved later.

In [ ]:
import chromadb

print("ChromaDB imported successfully.")

---
# 11. Create the RAG Collection

Create a ChromaDB collection that will store the `document embeddings`, `text`, and `metadata` for the **InvestIQ RAG system**.

In [ ]:
CHROMA_PATH = "/content/drive/MyDrive/data_handling_and_machine_learning/chroma_database"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(name="investiq_rag")

print("Persistent ChromaDB collection ready.")
print("Collection name:", collection.name)

---
## 11.1 Check the ChromaDB Collection

Verify that the persistent RAG collection currently contains no documents before inserting the 322 chunks.

In [ ]:
print("Collection name:", collection.name)
print("Number of records:", collection.count())

### 11.1.1 Prepare ChromaDB Records

Each ChromaDB record will contain:

- the chunk text as the document
- the embedding vector
- the standardized metadata
- a unique chunk ID

In [ ]:
sample_record = all_chunks[0]

print("Document:")
print(sample_record["text"][:500])

print("\nEmbedding dimensions:")
print(len(all_embeddings[0]))

print("\nMetadata:")
print(sample_record["metadata"])

## 11.2 Store the RAG Chunks in ChromaDB

Store each chunk together with its embedding and metadata.

The `chunk ID` is used as the unique identifier for each ChromaDB record.

In [ ]:
ids = [chunk["metadata"]["chunk_id"] for chunk in all_chunks]

documents = [chunk["text"] for chunk in all_chunks]

metadatas = [chunk["metadata"] for chunk in all_chunks]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=all_embeddings.tolist(),
    metadatas=metadatas
)

print("Chunks stored successfully.")
print("Number of records:", collection.count())

# 12. Test Semantic Retrieval

Before connecting the language model, test whether ChromaDB can retrieve relevant source chunks for a user question.

The question will first be converted into an embedding. ChromaDB will then compare that embedding with the stored chunk embeddings and return the most relevant chunks.

In [ ]:
query = "What does Lockheed Martin do?"

query_embedding = embedding_model.encode(query).tolist()

# Retrieval
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Retrieved chunks:", len(results["documents"][0]))

for i, document in enumerate(results["documents"][0], start=1):
    print(f"\n========== Result {i} ==========")
    print("Document:")
    print(document[:1000])

    print("\nMetadata:")
    print(results["metadatas"][0][i - 1])

## 12.1 Inspect Retrieval Distances

Inspect the similarity distance returned by ChromaDB for each retrieved chunk.

This helps us understand how strongly each chunk matched the query.

In [ ]:
print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n========== Result {i} ==========")
    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

## 12.2 Test Retrieval for Risk Information

Test whether semantic retrieval can identify the Risk Factors section when the question concerns risks faced by Lockheed Martin.

In [ ]:
query = "What risks does Lockheed Martin face?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n========== Result {i} ==========")
    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])
    print("\nDocument preview:")
    print(document[:500])

### 12.2.1 Inspect a Stored Risk Factors Chunk

Verify that a Lockheed Martin Risk Factors chunk is present in ChromaDB and inspect its content and metadata.

In [ ]:
risk_check = collection.get(ids=["LMT_Risk_Factors_001"])

print("Retrieved records:", len(risk_check["documents"]))

print("\nDocument:")
print(risk_check["documents"][0][:1000])

print("\nMetadata:")
print(risk_check["metadatas"][0])

## 12.3 Test Retrieval with a Specific Risk Query

Use a more specific question that closely matches the content of the Risk Factors section.

This helps determine whether retrieval quality depends on query specificity.

In [ ]:
query = "What risks are related to Lockheed Martin's reliance on U.S. government contracts?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n========== Result {i} ==========")
    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])
    print("\nDocument preview:")
    print(document[:500])

# 13. Create a Retrieval Evaluation Set

Test semantic retrieval across the four main 10-K sections:

- Business
- Risk Factors
- MD&A
- Financial Statements

The expected section is used only to evaluate whether retrieval returned the appropriate source section.

In [ ]:
retrieval_tests = [
    {
        "question": "What does Lockheed Martin do?",
        "expected_section": "Business"
    },
    {
        "question": "What risks does Lockheed Martin face from relying on U.S. government contracts?",
        "expected_section": "Risk Factors"
    },
    {
        "question": "What factors affected Lockheed Martin's financial performance?",
        "expected_section": "MD&A"
    },
    {
        "question": "What were Lockheed Martin's total assets?",
        "expected_section": "Financial Statements"
    }
]

for test in retrieval_tests:
    print("Question:", test["question"])
    print("Expected section:", test["expected_section"])
    print()

In [ ]:
for test in retrieval_tests:
    query = test["question"]
    expected_section = test["expected_section"]

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_sections = results["metadatas"][0]

    print("\n" + "=" * 70)
    print("Question:", query)
    print("Expected section:", expected_section)

    for i, metadata in enumerate(retrieved_sections, start=1):
        print(
            f"Result {i}: "
            f"{metadata['section']} | "
            f"{metadata['chunk_id']} | "
            f"Distance: {results['distances'][0][i-1]:.4f}"
        )

    found = any(
        metadata["section"] == expected_section
        for metadata in retrieved_sections
    )

    print("Expected section in top 3:", "YES ✅" if found else "NO ❌")

## 13.1. Inspect Retrieved Evidence

Inspect the text retrieved for a financial question and verify that the relevant factual evidence is present in the retrieved chunk.

In [ ]:
query = "What were Lockheed Martin's total assets?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:2000])

## 13.2 Inspect Retrieved Risk Evidence

Inspect the retrieved chunks for a question about Lockheed Martin's reliance on U.S. government contracts.

Verify whether the retrieved Risk Factors chunk contains evidence relevant to the question.

In [ ]:
query = "What risks does Lockheed Martin face from relying on U.S. government contracts?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

## 13.3 Inspect the MD&A retrieval


In [ ]:
query = "What factors affected Lockheed Martin's financial performance?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

In [ ]:
query = "According to Management’s Discussion and Analysis, what factors affected Lockheed Martin's financial performance?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print("Query:", query)

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

# 14. Test Metadata-Filtered Retrieval

The previous retrieval tests showed that semantic search can retrieve financially relevant chunks, but it does not always retrieve the section requested by the question.

Because each chunk already contains section metadata, I will now restrict the search to the MD&A section and test whether this improves retrieval of MD&A content.

This experiment keeps the embedding model, chunking strategy, and ChromaDB collection unchanged.

In [ ]:
query = "What factors affected Lockheed Martin's financial performance?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"section": "MD&A"}
)

print("Query:", query)
print("Filter: section = MD&A")

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

## 14.1 Test Risk Factors Metadata Filtering

The previous experiment showed that filtering the ChromaDB search by the `section` metadata successfully retrieved relevant MD&A chunks.

I will now apply the same approach to the Risk Factors section and check whether the retrieved chunks contain relevant risk information.

In [ ]:
query = "What risks does Lockheed Martin face from relying on U.S. government contracts?"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"section": "Risk Factors"}
)

print("Query:", query)
print("Filter: section = Risk Factors")

for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", distance)
    print("Company:", metadata["company"])
    print("Section:", metadata["section"])
    print("Chunk ID:", metadata["chunk_id"])

    print("\nRetrieved text:")
    print(document[:1500])

# 15. Define the Retrieval Strategy

The retrieval experiments showed that unrestricted semantic search can retrieve relevant information, but semantically similar chunks from unrelated sections may rank higher than the section that is most appropriate for the question.

Metadata filtering can restrict retrieval to a relevant 10-K section and improve the relevance of the retrieved context.

The RAG system will therefore use two retrieval modes:

1. Metadata-filtered retrieval when the relevant section is known.
2. Unrestricted semantic retrieval when no specific section can be identified.

The retrieved chunks will then be passed to the language model as context for answer generation.

# 16. Create a Reusable Retrieval Function

The retrieval experiments above were performed using individual ChromaDB queries.

To reuse the same retrieval logic later in the RAG pipeline, I will create a function that supports both retrieval modes:

- filtered retrieval when a section is specified
- unrestricted retrieval when no section is specified

The function will return the retrieved documents, metadata, and distances.

In [ ]:
def retrieve_chunks(query, n_results=3, section=None):
    query_embedding = embedding_model.encode(query).tolist()

    if section is not None:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where={"section": section}
        )
    else:
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )

    retrieved_chunks = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved_chunks.append({
            "text": document,
            "metadata": metadata,
            "distance": distance
        })

    return retrieved_chunks

## 16.1 Test the Retrieval Function

I will test the reusable retrieval function using a Risk Factors question and restrict the search to the Risk Factors section.

The result should match the behavior of the earlier metadata-filtered retrieval experiment.

In [ ]:
query = "What risks does Lockheed Martin face from relying on U.S. government contracts?"

retrieved_chunks = retrieve_chunks(
    query=query,
    n_results=3,
    section="Risk Factors"
)

print("Number of retrieved chunks:", len(retrieved_chunks))

for i, chunk in enumerate(retrieved_chunks, start=1):
    print(f"\n{'=' * 70}")
    print(f"Result {i}")
    print(f"{'=' * 70}")

    print("Distance:", chunk["distance"])
    print("Company:", chunk["metadata"]["company"])
    print("Section:", chunk["metadata"]["section"])
    print("Chunk ID:", chunk["metadata"]["chunk_id"])

    print("\nRetrieved text:")
    print(chunk["text"][:500])

# 17. Build the Retrieved Context

The retrieval step gives us a list of relevant chunks.

Before sending these chunks to the language model, we need to combine their text into a single context block.

This context will later be provided to Qwen together with the user's question.

For now, I will only build and inspect the context. I will not generate an answer with Qwen yet.

In [ ]:
context_parts = []

for i, chunk in enumerate(retrieved_chunks, start=1):
    context_parts.append(
        f"[Retrieved Source {i}]\n"
        f"Company: {chunk['metadata']['company']}\n"
        f"Section: {chunk['metadata']['section']}\n"
        f"Chunk ID: {chunk['metadata']['chunk_id']}\n"
        f"Text:\n{chunk['text']}"
    )

retrieved_context = "\n\n".join(context_parts)

print(retrieved_context[:5000])

# 18. Create a Reusable Context Builder

The retrieved chunks will eventually be passed to the language model.

To keep the RAG pipeline modular, I will create a reusable function that converts retrieved chunks into a formatted context string.

This function will preserve the source metadata together with the retrieved text.

In [ ]:
def build_retrieved_context(retrieved_chunks):
    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Retrieved Source {i}]\n"
            f"Company: {chunk['metadata']['company']}\n"
            f"Section: {chunk['metadata']['section']}\n"
            f"Chunk ID: {chunk['metadata']['chunk_id']}\n"
            f"Text:\n{chunk['text']}"
        )

    return "\n\n".join(context_parts)

## 18.1 Test the Context Builder

I will use the retrieved chunks from the previous retrieval test and pass them through the new context-building function.

The resulting context should contain the retrieved source information and text in the same order as before.

In [ ]:
retrieved_context = build_retrieved_context(retrieved_chunks)

print(retrieved_context[:5000])

# 19. Build the RAG Prompt

The RAG prompt combines the user's question with the retrieved source context.

The prompt instructs the language model to:

- answer using the retrieved context
- prioritize information supported by the provided sources
- avoid inventing information that is not supported by the context
- indicate when the retrieved information is insufficient
- identify the retrieved source used for the answer

At this stage, the prompt is only constructed and inspected. It will not yet be sent to Qwen.

In [ ]:
question = "What risks does Lockheed Martin face from relying on U.S. government contracts?"

rag_prompt = f"""
    You are an investment research assistant.

    Answer the user's question using the retrieved source context below.

    Rules:
    - Use the retrieved context as the primary source of evidence.
    - Do not invent facts that are not supported by the retrieved context.
    - If the retrieved context is insufficient to answer the question, say that the available information is insufficient.
    - Identify the source chunk IDs that support your answer.

    User question:
    {question}

    Retrieved context:
    {retrieved_context}
    """

print(rag_prompt[:6000])

# 20. Define the RAG Response Format

The RAG response should be structured so that the application can use the generated information consistently.

The model will be asked to return:

- an answer to the user's question
- the source chunk IDs supporting the answer
- an indication of whether the retrieved context was sufficient

This structure will later make it easier to display the answer and its sources in the Streamlit application.

In [ ]:
rag_prompt = f"""
You are an investment research assistant.

Answer the user's question using the retrieved source context below.

Rules:
- Use the retrieved context as the primary source of evidence.
- Do not invent facts that are not supported by the retrieved context.
- If the retrieved context is insufficient to answer the question, say that the available information is insufficient.
- Only cite source chunk IDs that actually appear in the retrieved context.
- Return your response as valid JSON using exactly these fields:
  "answer"
  "sources"
  "sufficient_context"

The "sources" field must contain a list of source chunk IDs.
The "sufficient_context" field must be either true or false.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

print(rag_prompt[:6000])

# 21 Load the Fine-Tuned Qwen Model

The RAG pipeline will use the fine-tuned InvestIQ Qwen adapter created during the fine-tuning stage.

The adapter is already saved in Google Drive, so the model does not need to be trained again.

I will first import the libraries required for inference and then reload the base Qwen model together with the fine-tuned adapter.

# 22. Reload the Fine-Tuned InvestIQ Model

The fine-tuned model consists of:

- the original Qwen2.5-1.5B-Instruct base model
- the InvestIQ LoRA adapter trained earlier

The adapter is stored separately, so I will load the base model first and then attach the adapter.

This makes the RAG notebook independent from the previous training notebook.

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
else:
    print("Qwen inference must use CPU in this session.")

In [ ]:
import inspect

print(inspect.getsource(generate_qwen_response))

In [ ]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/week_04/outputs/Finetuned_version01/InvestIQ_finetuned_adapter_updated")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="cuda",
    torch_dtype=torch.float16
)

base_model.config.use_cache = True

trained_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

trained_model.eval()

In [ ]:
print("Model device:", next(trained_model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
objects_to_check = [
    "ups_uber_collection",
    "embedding_model",
    "reranker"
]

for name in objects_to_check:
    if name in globals():
        print(f"{name}: available")
    else:
        print(f"{name}: NOT available")

# 23. Generate the RAG Answer

The fine-tuned InvestIQ model has now been successfully reloaded.

I will provide it with the RAG prompt containing:

- the user's question
- the retrieved 10-K evidence
- instructions for source-grounded JSON output

The generated response will be inspected before we make any changes to the pipeline.

In [ ]:
messages = [
    {
        "role": "user",
        "content": rag_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(rag_response)

# 24. Test RAG Generation Without Structured JSON

The previous generation test required the model to produce a strict JSON response.

The model followed the requested structure but returned an empty answer.

To determine whether the problem is caused by the RAG context or by the structured-output instruction, I will temporarily remove the JSON requirement.

The model will instead be asked to answer the question directly using only the retrieved context.

This is a diagnostic experiment and does not change the final RAG design yet.

In [ ]:
diagnostic_prompt = f"""
You are an investment research assistant.

Answer the user's question using only the retrieved source context below.

Rules:
- Use the retrieved context as the primary source of evidence.
- Do not invent facts that are not supported by the retrieved context.
- If the retrieved context is insufficient, clearly say so.
- Explain the relevant risks using the information provided in the sources.
- Mention the source chunk IDs that support your answer.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

messages = [
    {
        "role": "user",
        "content": diagnostic_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

diagnostic_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(diagnostic_response)

## 24.1 Evaluate the First RAG Answer

The generated answer is coherent, but some statements are not directly supported by the retrieved context.

I will evaluate the response against the retrieved evidence before modifying the RAG prompt.

The purpose of this test is to distinguish:

- information supported by the retrieved documents
- information added by the language model
- information that contradicts the retrieved documents

---
### Grounding Evaluation

| Generated claim | Supported by retrieved context? | Observation |
|---|---|---|
| Lockheed Martin depends heavily on U.S. Government contracts | Yes | Directly stated in `LMT_Risk_Factors_001` |
| Changes in government priorities or spending can affect the business | Yes | Directly supported |
| Government contracts involve extensive laws and regulations | Yes | Directly supported by `LMT_Risk_Factors_003` |
| Non-compliance can result in fines, penalties, termination, suspension or debarment | Yes | Directly supported |
| Government procurement policies can change and affect profitability/business | Yes | Directly supported |
| Costs can increase and margins can be reduced | Yes | Directly supported |
| Contracts have restrictions on termination without cause | No / misleading | The retrieved text says the U.S. Government can terminate contracts for convenience |
| Limitations on subcontractor liability | No | Not present in the retrieved context |
| Confidentiality, integrity and availability standards | No | Not established by the retrieved context |
| Source chunk IDs were provided in the answer | No | The model was instructed to provide them but did not |

## 24.2 Strengthen the RAG Grounding Instructions

The first generation test produced a useful answer but also included several claims that were not supported by the retrieved context.

I will therefore strengthen the grounding instructions.

The model must treat the retrieved context as the evidence boundary for the answer:

- factual claims must be supported by the retrieved context
- unsupported claims must be omitted
- the model should not fill missing information using its general knowledge
- source chunk IDs should correspond to the evidence used

This is a prompt-level change only. The model and retrieval system remain unchanged.

In [ ]:
grounded_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim in your answer must be supported by the retrieved context.
- Do not use your general knowledge to add facts that are not present in the retrieved context.
- If a detail is not supported by the retrieved context, leave it out.
- Do not reinterpret or contradict the retrieved source text.
- If the retrieved context is insufficient to answer part of the question, explicitly state that the available context is insufficient.
- After each major factual statement, identify the source chunk ID that supports it.
- Use only source chunk IDs that actually appear in the retrieved context.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

print(grounded_prompt[:6000])

## 24.3 Test the Strengthened Grounding Prompt

I will now generate an answer using the strengthened grounding instructions.

The model, retrieved chunks, generation settings, and retrieval method remain unchanged.

Only the prompt instructions have changed.

This allows us to compare the result with the previous diagnostic generation.

In [ ]:
messages = [
    {
        "role": "user",
        "content": grounded_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

grounded_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(grounded_response)

## 24.4 Test a Simple Grounded Answer Format

The previous grounding prompt reduced unsupported claims, but the model did not include the requested source chunk IDs.

In this test, I will use a simpler output structure:

- Answer
- Supporting Sources

The retrieval method, retrieved chunks, model, and generation settings remain unchanged.

This helps test whether a simpler response format improves source attribution without introducing the stricter JSON format that previously caused an empty answer.

In [ ]:
source_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim must be supported by the retrieved context.
- Do not add facts from your general knowledge.
- If a detail is not supported by the retrieved context, leave it out.
- Do not reinterpret or contradict the retrieved source text.
- If the context is insufficient, clearly state that.

Use exactly this response structure:

Answer:
<Give a concise answer supported only by the retrieved context.>

Supporting Sources:
<List only the chunk IDs that directly support the answer.>

User question:
{question}

Retrieved context:
{retrieved_context}
"""

print(source_prompt[:6000])

# 25. Generate a Grounded Answer with Fine-Tuned Qwen

The previous step constructed and inspected the prompt but did not run the language model.

In this step, the prompt will be tokenized and passed to the **fine-tuned Qwen model for inference**.

Inference means using an already-trained model to generate a prediction or response for new input. The model weights are not updated during this process.

The generation settings remain the same as in the previous grounding experiment so that the results can be compared fairly.

In [ ]:
messages = [
    {
        "role": "user",
        "content": source_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

source_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(source_response)

## 25.1 Evaluate Answer Grounding

I will manually compare the generated answer with the retrieved source chunks.

The goal is to determine whether each factual claim in the generated answer is supported by the retrieved evidence.

This evaluates the generation stage separately from retrieval.

### Grounding Evaluation

| Generated claim | Supported by retrieved context? | Supporting chunk |
|---|---|---|
| Lockheed Martin faces risks from delays or reductions in U.S. Government spending. | Yes | LMT_Risk_Factors_001 |
| Supplier dependency can increase costs and create schedule delays. | Yes | LMT_Risk_Factors_003 |
| Lockheed Martin faces compliance-related risks. | Yes | LMT_Risk_Factors_003 |
| These risks can affect the company's business and competitiveness. | Yes | LMT_Risk_Factors_001 / LMT_Risk_Factors_003 |
| These risks can affect the company's financial stability. | Partially supported / not explicit in retrieved context | — |

---

# 26. Compare Fine-Tuned Qwen With and Without RAG

I will compare the fine-tuned Qwen model's response **with** and **without** retrieved source context.

The purpose is to observe how retrieval changes the information available to the model.

The same user question will be used for both tests.

The RAG version receives the retrieved 10-K evidence, while the non-RAG version does not.

---
## 26.1 Fine-Tuned Qwen Without RAG
---

In [ ]:
no_rag_prompt = f"""
You are an investment research assistant.

Answer the user's question as accurately as possible.

User question:
{question}
"""

messages = [
    {
        "role": "user",
        "content": no_rag_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

no_rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(no_rag_response)

In [ ]:
probability_question = """
The model estimates a 72% probability of a positive 1-year return for a company.
The company also has declining revenue growth and increasing debt.
How should I interpret these signals together?
"""

probability_prompt = f"""
You are an investment research assistant.

Answer the user's question as accurately as possible.

User question:
{probability_question}
"""

messages = [
    {"role": "user", "content": probability_prompt}
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

probability_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(probability_response)

---
## 26.2. Generate an Answer With RAG
---

The previous experiment generated an answer using the fine-tuned Qwen model without external retrieved context.

Now I will provide the same model with relevant chunks retrieved from the InvestIQ ChromaDB collection.

The goal is to observe how the retrieved 10-K evidence changes the answer.

The comparison uses:

- the same fine-tuned Qwen model
- the same user question
- the same generation settings
- retrieved evidence from the Lockheed Martin 2025 10-K

In [ ]:
question = "What risks does Lockheed Martin face from relying on U.S. government contracts?"

retrieved_chunks = retrieve_chunks(
    question,
    n_results=3,
    section="Risk Factors"
)

retrieved_context = build_retrieved_context(retrieved_chunks)

print(retrieved_context)

## 26.3 Generate a Grounded RAG Answer

The retrieved Risk Factors chunks provide evidence about Lockheed Martin's dependence on U.S. Government contracts, procurement requirements, regulatory compliance, and changes in government contracting policies.

I will now provide these retrieved chunks to the fine-tuned Qwen model and ask it to answer the original question using the evidence.

The model must treat the retrieved context as the evidence boundary and should not add unsupported facts.

In [ ]:
rag_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Grounding rules:
- Treat the retrieved context as the evidence boundary for your answer.
- Every factual claim must be supported by the retrieved context.
- Do not add facts from your general knowledge.
- If a detail is not supported by the retrieved context, leave it out.
- Do not contradict or reinterpret the source text.
- Mention the source chunk IDs that support your answer.

User question:
{question}

Retrieved context:
{retrieved_context}
"""

messages = [
    {"role": "user", "content": rag_prompt}
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(rag_response)

## 26.4 Compare RAG and Non-RAG Responses

The same fine-tuned Qwen model was asked the same question in two conditions:

**1. Without RAG** — Qwen answered using its learned behavior and existing model knowledge.

**2. With RAG** — Qwen received three relevant Lockheed Martin Risk Factors chunks from the 2025 10-K.

I will compare the responses based on:

- use of company-specific evidence
- factual grounding
- specificity
- source attribution
- unsupported claims
- response structure

In [ ]:
print("=" * 80)
print("WITHOUT RAG")
print("=" * 80)
print(no_rag_response)

print("\n" + "=" * 80)
print("WITH RAG")
print("=" * 80)
print(rag_response)

---
### Evaluate RAG Grounding

The RAG response is evaluated against the retrieved Lockheed Martin Risk Factors chunks.

The purpose is to determine whether the generated claims are supported by the retrieved evidence.

This evaluation focuses on factual grounding rather than whether one response is generally "better" than the other.

---

In [ ]:
rag_grounding_evaluation = [
    {
        "claim": "Lockheed Martin depends heavily on U.S. Government contracts.",
        "supported": "Yes",
        "source": "LMT_Risk_Factors_001"
    },
    {
        "claim": "Changes in U.S. defense spending, budget uncertainty, or government shutdowns can negatively affect the business.",
        "supported": "Yes",
        "source": "LMT_Risk_Factors_001"
    },
    {
        "claim": "Delayed or insufficient Congressional funding can lead to reduced orders, cancellations, or disruptions.",
        "supported": "Yes",
        "source": "LMT_Risk_Factors_001"
    },
    {
        "claim": "Changes in U.S. Government procurement policies can affect profitability and the ability to win new business.",
        "supported": "Yes",
        "source": "LMT_Risk_Factors_007"
    },
    {
        "claim": "OTAs are not subject to all traditional procurement laws.",
        "supported": "Partially",
        "source": "LMT_Risk_Factors_007"
    },
    {
        "claim": "OTAs increase Lockheed Martin's exposure to regulatory changes and reduce predictability.",
        "supported": "Not explicit",
        "source": "—"
    },
    {
        "claim": "Government-contract-related risks can affect profitability and cash flow.",
        "supported": "Yes",
        "source": "LMT_Risk_Factors_007"
    }
]

for item in rag_grounding_evaluation:
    print(f"Claim: {item['claim']}")
    print(f"Supported: {item['supported']}")
    print(f"Source: {item['source']}")
    print()

# 27. Test Structured RAG Output

The previous RAG response used the retrieved evidence successfully, but it did not reliably return the requested source chunk IDs.

For the final InvestIQ application, structured output would make it easier for Streamlit to display:

- the generated answer
- the supporting source chunks
- whether sufficient evidence was retrieved

I will therefore test whether the fine-tuned model can follow a simple JSON response structure when RAG context is provided.

In [ ]:
structured_rag_prompt = f"""
You are an investment research assistant.

Answer the user's question using ONLY the retrieved source context below.

Rules:
- Treat the retrieved context as the evidence boundary.
- Every factual claim must be supported by the retrieved context.
- Do not add facts from general knowledge.
- If the context does not support a detail, leave that detail out.
- Do not invent source IDs.
- Only use source IDs that appear in the retrieved context.

Return ONLY valid JSON using exactly this structure:

{{
  "answer": "A concise answer supported by the retrieved context.",
  "sources": ["source_id_1", "source_id_2"]
}}

User question:
{question}

Retrieved context:
{retrieved_context}
"""

messages = [
    {"role": "user", "content": structured_rag_prompt}
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

structured_rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(structured_rag_response)

# 28. Create the Reusable RAG Function

The RAG experiments are now complete enough to combine into one reusable function.

The function will:

1. Receive a user's question.
2. Retrieve relevant chunks from ChromaDB.
3. Optionally restrict retrieval to a specific 10-K section.
4. Build the retrieved context.
5. Send the question and evidence to the fine-tuned Qwen model.
6. Return the generated answer and source chunk IDs.

This separates the RAG pipeline from the notebook experiments and makes it easier to integrate into the final InvestIQ application later.

In [ ]:
def generate_qwen_response(prompt, max_new_tokens=400):
    messages = [
        {"role": "user", "content": prompt}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(trained_model.device)

    with torch.no_grad():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

## 28.1 Create the RAG Pipeline Function

The `generate_qwen_response()` helper handles Qwen generation.

The next function connects the complete RAG workflow:

```text
User question
→ retrieve relevant chunks
→ build source-grounded context
→ create the RAG prompt
→ generate the answer with Qwen
```

The function will return both the generated answer and the retrieved source information so that the final application can display the evidence used.

In [ ]:
def answer_with_rag(question, n_results=3, section=None):

    # Step 1: Retrieve relevant chunks
    retrieved_chunks = retrieve_chunks(
        question,
        n_results=n_results,
        section=section
    )

    # Step 2: Build context from retrieved chunks
    retrieved_context = build_retrieved_context(retrieved_chunks)

    # Step 3: Build the grounded prompt
    rag_prompt = f"""
    You are an investment research assistant.

    Answer the user's question using ONLY the retrieved source context below.

    Grounding rules:
    - Treat the retrieved context as the evidence boundary.
    - Every factual claim must be supported by the retrieved context.
    - Do not add facts from general knowledge.
    - If the context does not support a detail, leave it out.
    - Do not invent source IDs.
    - Only use source IDs that appear in the retrieved context.

    Return ONLY valid JSON using exactly this structure:

    {{
      "answer": "A concise answer supported by the retrieved context.",
      "sources": ["source_id_1", "source_id_2"]
    }}

    User question:
    {question}

    Retrieved context:
    {retrieved_context}
    """

    # Step 4: Generate the answer
    response = generate_qwen_response(rag_prompt)

    # Step 5: Return answer and retrieval information
    return {
        "response": response,
        "retrieved_chunks": retrieved_chunks
    }

## 28.1.1 Test the Complete RAG Pipeline

The complete RAG pipeline is now wrapped in `answer_with_rag()`.

I will test it using the same Lockheed Martin question used in the previous retrieval and grounding experiments.

The retrieval scope will be restricted to the Risk Factors section because the question explicitly asks about risks.

In [ ]:
test_result = answer_with_rag(
    question="What risks does Lockheed Martin face from relying on U.S. government contracts?",
    n_results=3,
    section="Risk Factors"
)

print(test_result["response"])

# 29. Test Insufficient Retrieved Context

A grounded RAG system should not generate an answer merely because the underlying language model may know something about the topic.

If the retrieved source documents do not contain enough information to answer a question, the system should acknowledge that the available context is insufficient.

This test checks that behavior.

In [ ]:
insufficient_question = "What university did Lockheed Martin's CEO attend?"

insufficient_result = answer_with_rag(
    question=insufficient_question,
    n_results=3
)

print(insufficient_result["response"])

## 29.1. Strengthen RAG Grounding for Insufficient Context

The previous test showed that the model can still answer from its pretrained knowledge when the retrieved context does not contain the requested information.

We will strengthen the prompt by explicitly requiring the model to distinguish between:

1. Information supported by the retrieved sources
2. Information known by the language model but not present in the retrieved sources

For RAG, only the first category may be used in the answer.

In [ ]:
rag_prompt = f"""
You are an investment research assistant using a retrieval-augmented generation system.

IMPORTANT:
The retrieved source context is the ONLY source of factual information you may use.

You must NOT answer from your pretrained knowledge.

Grounding rules:
- Use only information explicitly supported by the retrieved context.
- Do not infer missing facts.
- Do not use outside knowledge, even if you believe you know the answer.
- If the retrieved context does not contain the information needed to answer the question, say that the available retrieved sources are insufficient.
- Never provide a guessed or remembered answer when the retrieved context is insufficient.
- Every factual statement in "answer" must be directly supported by the retrieved context.
- Do not invent source IDs.
- Only use source IDs that appear in the retrieved context.
- If the answer cannot be supported, return an empty sources list.

Return ONLY valid JSON using exactly this structure:

{{
  "answer": "A concise answer supported only by the retrieved context, or a statement that the retrieved sources are insufficient.",
  "sources": ["source_id_1", "source_id_2"]
}}

User question:
{question}

Retrieved context:
{retrieved_context}
"""

In [ ]:
insufficient_result = answer_with_rag(
    question="What university did Lockheed Martin's CEO attend?",
    n_results=3
)

print(insufficient_result["response"])

## 29.2 Inspect Retrieved Context for an Insufficient-Information Query

Before changing the RAG generation strategy, we will inspect the documents retrieved for the insufficient-information question.

This lets us verify whether the requested information is actually absent from the retrieved context provided to the language model.

In [ ]:
insufficient_question = "What university did Lockheed Martin's CEO attend?"

insufficient_result = answer_with_rag(
    question=insufficient_question,
    n_results=3
)

for i, chunk in enumerate(insufficient_result["retrieved_chunks"], start=1):
    print(f"\n--- Retrieved Source {i} ---")
    print("Chunk ID:", chunk["metadata"]["chunk_id"])
    print("Section:", chunk["metadata"]["section"])
    print("Distance:", chunk["distance"])
    print("Text preview:")
    print(chunk["text"][:1000])

## 29.3 Test Qwen with Explicitly Empty Retrieved Context

The previous test retrieved real but irrelevant financial and business information.

We will now provide the model with no retrieved evidence at all.

This isolates the generation behavior from the retrieval system.

In [ ]:
empty_context_prompt = """
You are an investment research assistant using a retrieval-augmented generation system.

The retrieved source context below is the ONLY source of factual information you may use.

If the retrieved context does not contain enough information to answer the question:
- Do not use your pretrained knowledge.
- Do not guess.
- Do not infer the answer.
- State clearly that the available retrieved context is insufficient.

Return ONLY valid JSON using exactly this structure:

{
  "answer": "A concise grounded answer, or a statement that the retrieved context is insufficient.",
  "sources": []
}

User question:
What university did Lockheed Martin's CEO attend?

Retrieved context:
NO RELEVANT INFORMATION WAS RETRIEVED.
"""

empty_context_response = generate_qwen_response(
    empty_context_prompt
)

print(empty_context_response)

## 29.4 Compare Retrieval Distances

We will compare the ChromaDB retrieval distances for:

1. A question that is clearly answerable from the documents.
2. A question whose answer is not present in the retrieved documents.

The goal is to understand whether retrieval distance can help us identify insufficient retrieval before sending context to Qwen.

In [ ]:
comparison_questions = [
    "What were Lockheed Martin's total assets?",
    "What university did Lockheed Martin's CEO attend?"
]

for question in comparison_questions:

    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("=" * 70)

    retrieved_chunks = retrieve_chunks(
        question,
        n_results=3
    )

    for i, chunk in enumerate(retrieved_chunks, start=1):
        print(
            f"{i}. "
            f"{chunk['metadata']['chunk_id']} | "
            f"section={chunk['metadata']['section']} | "
            f"distance={chunk['distance']:.4f}"
        )

# 30. Build a Retrieval Relevance Test Set

We will create a small evaluation set containing questions that are:

- answerable from the RAG documents
- not answerable from the RAG documents

For each question, we will inspect the distance of the most similar retrieved chunk.

This is an exploratory evaluation, not a formal accuracy benchmark.

In [ ]:
retrieval_relevance_tests = [
    {
        "question": "What does Lockheed Martin do?",
        "expected": "answerable"
    },
    {
        "question": "What risks does Lockheed Martin face from relying on U.S. government contracts?",
        "expected": "answerable"
    },
    {
        "question": "What factors affected Lockheed Martin's financial performance?",
        "expected": "answerable"
    },
    {
        "question": "What were Lockheed Martin's total assets?",
        "expected": "answerable"
    },
    {
        "question": "What university did Lockheed Martin's CEO attend?",
        "expected": "not_answerable"
    },
    {
        "question": "What is Lockheed Martin's CEO's favorite food?",
        "expected": "not_answerable"
    }
]

for test in retrieval_relevance_tests:

    retrieved_chunks = retrieve_chunks(
        test["question"],
        n_results=3
    )

    top_chunk = retrieved_chunks[0]

    print(
        f"{test['expected']:>13} | "
        f"distance={top_chunk['distance']:.4f} | "
        f"{test['question']}"
    )

## 30.1 Visualize Retrieval Distances

We will visualize the top retrieval distance for each test question.

This helps us see whether answerable and non-answerable questions form clearly separated groups.

This is only an exploratory visualization because the evaluation set is very small.

In [ ]:
import matplotlib.pyplot as plt

labels = [
    "Do",
    "Risks",
    "Performance",
    "Assets",
    "CEO university",
    "CEO favorite food"
]

distances = [
    0.9687,
    0.8841,
    0.7902,
    0.6394,
    1.1457,
    1.0793
]

plt.figure(figsize=(10, 5))

plt.bar(labels, distances)

plt.ylabel("ChromaDB distance")
plt.xlabel("Question")
plt.title("Top Retrieval Distance by Test Question")

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# 31. Inspect Top Retrieved Chunks

Retrieval distance provides a similarity signal, but similarity does not guarantee that a retrieved chunk contains the answer.

We will inspect the complete top-ranked chunk for:

1. An answerable question.
2. A question whose answer is not present in the documents.

This helps us understand what evidence is actually being passed to the generation model.

In [ ]:
inspection_questions = [
    "What were Lockheed Martin's total assets?",
    "What university did Lockheed Martin's CEO attend?"
]

for question in inspection_questions:

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    retrieved_chunks = retrieve_chunks(
        question,
        n_results=1
    )

    chunk = retrieved_chunks[0]

    print("Chunk ID:", chunk["metadata"]["chunk_id"])
    print("Section:", chunk["metadata"]["section"])
    print("Distance:", chunk["distance"])
    print("\nRetrieved text:\n")
    print(chunk["text"])

## 31.1. Test Evidence Support Classification

Semantic similarity alone does not guarantee that retrieved text supports an answer.

We will test a separate evidence-checking step.

The checker will receive:
- the user's question
- the retrieved context

It will classify the context as either:

- SUPPORTED
- NOT_SUPPORTED

It will not answer the question.

This experiment tests whether a separate classification step can detect insufficient evidence before answer generation.

In [ ]:
def check_evidence_support(question, retrieved_context):

    evidence_prompt = f"""
    You are an evidence checker for a retrieval-augmented generation system.

    Your task is NOT to answer the user's question.

    Determine whether the retrieved context contains enough explicit information
    to support an answer to the question.

    Rules:
    - Return SUPPORTED only if the retrieved context explicitly contains
      information that can answer the question.
    - Return NOT_SUPPORTED if the answer is missing.
    - Do not use your pretrained knowledge.
    - Do not infer missing information.
    - Semantic similarity alone is not enough.
    - Return exactly one label:
      SUPPORTED
      or
      NOT_SUPPORTED

    Question:
    {question}

    Retrieved context:
    {retrieved_context}

    Classification:
    """

    response = generate_qwen_response(
        evidence_prompt,
        max_new_tokens=20
    )

    return response.strip()

In [ ]:
evidence_test_questions = [
    "What were Lockheed Martin's total assets?",
    "What university did Lockheed Martin's CEO attend?"
]

for question in evidence_test_questions:

    retrieved_chunks = retrieve_chunks(
        question,
        n_results=3
    )

    retrieved_context = build_retrieved_context(retrieved_chunks)

    result = check_evidence_support(
        question,
        retrieved_context
    )

    print("\nQuestion:", question)
    print("Evidence check:", result)

## 31.2 Expand the Retrieval Relevance Test Set

The previous experiment showed that Qwen is not reliable as an evidence classifier.

We will therefore evaluate the retrieval system itself.

We will use several answerable and non-answerable questions and compare the top retrieval distances.

The goal is to determine whether retrieval distance can serve as a conservative signal for insufficient retrieval.

In [ ]:
expanded_retrieval_tests = [
    # Answerable questions
    {
        "question": "What does Lockheed Martin do?",
        "expected": "answerable"
    },
    {
        "question": "What were Lockheed Martin's total assets?",
        "expected": "answerable"
    },
    {
        "question": "What were Lockheed Martin's total liabilities?",
        "expected": "answerable"
    },
    {
        "question": "What was Lockheed Martin's net earnings in 2025?",
        "expected": "answerable"
    },
    {
        "question": "What risks does Lockheed Martin face from relying on U.S. government contracts?",
        "expected": "answerable"
    },
    {
        "question": "What factors affected Lockheed Martin's financial performance?",
        "expected": "answerable"
    },

    # Non-answerable questions
    {
        "question": "What university did Lockheed Martin's CEO attend?",
        "expected": "not_answerable"
    },
    {
        "question": "What is Lockheed Martin's CEO's favorite food?",
        "expected": "not_answerable"
    },
    {
        "question": "What is Lockheed Martin's CEO's favorite movie?",
        "expected": "not_answerable"
    },
    {
        "question": "What is Lockheed Martin's CEO's home address?",
        "expected": "not_answerable"
    }
]

for test in expanded_retrieval_tests:

    retrieved_chunks = retrieve_chunks(
        test["question"],
        n_results=3
    )

    top_chunk = retrieved_chunks[0]

    print(
        f"{test['expected']:>13} | "
        f"distance={top_chunk['distance']:.4f} | "
        f"{test['question']}"
    )

## 31.3. Build a Broader RAG Retrieval Evaluation

The previous tests suggest that the retrieval layer can distinguish some
answerable questions from questions outside the available document scope.

We will now test retrieval across the four RAG sections:

- Business
- Risk Factors
- MD&A
- Financial Statements

We will also include questions whose answers are not expected to be
contained in the 10-K documents.

This evaluation is exploratory and is intended to help us understand
retrieval behavior before introducing a relevance threshold.

In [ ]:
retrieval_evaluation = [
    # Business
    {
        "question": "What does Lockheed Martin do?",
        "expected_section": "Business"
    },
    {
        "question": "How many employees did Lockheed Martin have in 2025?",
        "expected_section": "Business"
    },

    # Risk Factors
    {
        "question": "What risks does Lockheed Martin face from relying on U.S. government contracts?",
        "expected_section": "Risk Factors"
    },
    {
        "question": "What risks can changes in government procurement policies create?",
        "expected_section": "Risk Factors"
    },

    # MD&A
    {
        "question": "What factors affected Lockheed Martin's financial performance?",
        "expected_section": "MD&A"
    },
    {
        "question": "What caused changes in Lockheed Martin's cash flows?",
        "expected_section": "MD&A"
    },

    # Financial Statements
    {
        "question": "What were Lockheed Martin's total assets?",
        "expected_section": "Financial Statements"
    },
    {
        "question": "What were Lockheed Martin's total liabilities?",
        "expected_section": "Financial Statements"
    },
    {
        "question": "What was Lockheed Martin's net earnings in 2025?",
        "expected_section": "Financial Statements"
    },

    # Outside expected document scope
    {
        "question": "What university did Lockheed Martin's CEO attend?",
        "expected_section": None
    },
    {
        "question": "What is Lockheed Martin's CEO's favorite food?",
        "expected_section": None
    },
    {
        "question": "What is Lockheed Martin's CEO's home address?",
        "expected_section": None
    }
]

for test in retrieval_evaluation:

    retrieved_chunks = retrieve_chunks(
        test["question"],
        n_results=3
    )

    top_chunk = retrieved_chunks[0]

    print(
        f"distance={top_chunk['distance']:.4f} | "
        f"retrieved_section={top_chunk['metadata']['section']} | "
        f"expected={test['expected_section']} | "
        f"{test['question']}"
    )

# 32. Test Section Routing

Our current retrieval searches all chunks in the collection.

This can cause questions about MD&A or Risk Factors to retrieve
semantically similar chunks from Financial Statements.

We will first test whether the embedding model can identify the most
appropriate 10-K section for a question.

At this stage, we are only testing the routing decision.
We are not changing the existing retrieval function yet.

In [ ]:
section_descriptions = {
    "Business": (
        "The company's business, products, services, operations, "
        "markets, customers, employees, strategy, and organization."
    ),
    "Risk Factors": (
        "Risks and uncertainties affecting the company, including "
        "business risks, government contracts, regulations, competition, "
        "markets, operations, and other potential threats."
    ),
    "MD&A": (
        "Management's discussion and analysis of financial condition "
        "and results of operations, including financial performance, "
        "revenue, expenses, cash flows, liquidity, and changes in results."
    ),
    "Financial Statements": (
        "Consolidated financial statements and related financial data, "
        "including assets, liabilities, equity, earnings, cash flows, "
        "balance sheets, income statements, and accounting information."
    )
}

section_names = list(section_descriptions.keys())

section_embeddings = embedding_model.encode(
    list(section_descriptions.values())
)

print("Section descriptions embedded:", len(section_embeddings))
print("Embedding dimension:", section_embeddings.shape[1])

## 32.1 Test Semantic Section Routing

We compare each question with the four section descriptions using cosine similarity.

A higher similarity means the question is semantically closer to that section.

We will test questions whose expected sections we already know from our earlier retrieval evaluation.

In [ ]:
def route_question_to_section(question):
    question_embedding = embedding_model.encode([question])

    similarities = cosine_similarity(
        question_embedding,
        section_embeddings
    )[0]

    ranked_sections = sorted(
        zip(section_names, similarities),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_sections


routing_tests = [
    ("What does Lockheed Martin do?", "Business"),
    ("How many employees did Lockheed Martin have in 2025?", "Business"),
    ("What risks does Lockheed Martin face from relying on U.S. government contracts?", "Risk Factors"),
    ("What risks can changes in government procurement policies create?", "Risk Factors"),
    ("What factors affected Lockheed Martin's financial performance?", "MD&A"),
    ("What caused changes in Lockheed Martin's cash flows?", "MD&A"),
    ("What were Lockheed Martin's total assets?", "Financial Statements"),
    ("What was Lockheed Martin's net earnings in 2025?", "Financial Statements"),
]

for question, expected_section in routing_tests:
    ranked = route_question_to_section(question)

    print("\nQuestion:", question)
    print("Expected:", expected_section)

    for section, similarity in ranked:
        print(f"  {section}: {similarity:.4f}")

    print("Predicted:", ranked[0][0])

## 32.2 Inspect Section Routing Confidence

The highest-scoring section is not always reliable.

We calculate the gap between the highest and second-highest section similarity.

A large gap indicates a clearer section match.
A small gap indicates that the question is ambiguous and should not necessarily be filtered to one section.

In [ ]:
for question, expected_section in routing_tests:
    ranked = route_question_to_section(question)

    top_section, top_score = ranked[0]
    second_section, second_score = ranked[1]

    gap = top_score - second_score

    print("\nQuestion:", question)
    print("Expected:", expected_section)
    print("Top section:", top_section, f"({top_score:.4f})")
    print("Second section:", second_section, f"({second_score:.4f})")
    print("Gap:", f"{gap:.4f}")

## 32.3 Refine Section Descriptions

The first routing test showed that broad section descriptions work well for conceptual questions but are weaker for specific accounting questions.

We will make the section descriptions more representative of the terminology commonly found in each section.

This changes only the section-routing layer. It does not change the source documents, chunks, or ChromaDB collection.

In [ ]:
section_descriptions = {
    "Business": (
        "The company's business, products, services, operations, "
        "markets, customers, employees, strategy, organization, "
        "business segments, and industry."
    ),

    "Risk Factors": (
        "Risks and uncertainties affecting the company, including "
        "business risks, government contracts, government spending, "
        "regulations, procurement policies, competition, markets, "
        "operations, cybersecurity, supply chains, and other threats."
    ),

    "MD&A": (
        "Management's discussion and analysis of financial condition "
        "and results of operations, including financial performance, "
        "revenue, sales, expenses, operating results, cash flows, "
        "liquidity, capital resources, changes in results, and trends."
    ),

    "Financial Statements": (
        "Consolidated financial statements and accounting information, "
        "including balance sheets, income statements, statements of "
        "cash flows, assets, total assets, liabilities, total liabilities, "
        "equity, revenue, net earnings, net income, expenses, "
        "accounting data, and financial statement notes."
    )
}

section_names = list(section_descriptions.keys())

section_embeddings = embedding_model.encode(
    list(section_descriptions.values())
)

print("Section descriptions embedded:", len(section_embeddings))
print("Embedding dimension:", section_embeddings.shape[1])

## 32.4 Compare Refined Section Routing

We re-run the same routing questions using the refined section descriptions.

The goal is to see whether adding accounting and financial terminology improves the routing of Financial Statements questions without harming the other sections.

In [ ]:
for question, expected_section in routing_tests:
    ranked = route_question_to_section(question)

    top_section, top_score = ranked[0]
    second_section, second_score = ranked[1]

    gap = top_score - second_score

    print("\nQuestion:", question)
    print("Expected:", expected_section)

    for section, similarity in ranked:
        print(f"  {section}: {similarity:.4f}")

    print("Predicted:", top_section)
    print("Gap:", f"{gap:.4f}")

## 32.5 Diagnose Retrieval Depth

Our earlier retrieval tests used only the top 3 results.

A relevant chunk may exist in the database but rank below the top 3.

We will retrieve the top 10 results for several questions to determine whether the problem is:

1. the relevant information is not being retrieved at all, or
2. the relevant information is retrieved but ranked too low.

In [ ]:
diagnostic_questions = [
    (
        "What risks does Lockheed Martin face from relying on U.S. government contracts?",
        "Risk Factors"
    ),
    (
        "What factors affected Lockheed Martin's financial performance?",
        "MD&A"
    ),
    (
        "What caused changes in Lockheed Martin's cash flows?",
        "MD&A"
    )
]

for question, expected_section in diagnostic_questions:

    results = retrieve_chunks(
        question,
        n_results=10
    )

    print("\n" + "=" * 80)
    print("Question:", question)
    print("Expected section:", expected_section)
    print("=" * 80)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. "
            f"{result['metadata']['chunk_id']} | "
            f"{result['metadata']['section']} | "
            f"distance={result['distance']:.4f}"
        )

In [ ]:
collection_count = collection.count()

print("Total chunks in collection:", collection_count)

sample = collection.get(
    limit=collection_count,
    include=["metadatas"]
)

company_counts = {}

for metadata in sample["metadatas"]:
    company = metadata["company"]
    company_counts[company] = company_counts.get(company, 0) + 1

print("\nChunks by company:")

for company, count in sorted(company_counts.items()):
    print(f"  {company}: {count}")

# 33. Create a Clean LMT + WBD Retrieval Collection

The shared ChromaDB collection currently contains documents from multiple team members.

For this baseline experiment, we will create a separate collection containing only:

- Lockheed Martin
- Warner Bros. Discovery

This allows us to evaluate retrieval without unrelated companies affecting the ranking.

The original shared collection will not be modified.

In [ ]:
clean_collection = chroma_client.get_or_create_collection(
    name="investiq_rag_lmt_wbd"
)

print("Clean collection count before insertion:", clean_collection.count())

## 33.1 Copy LMT and WBD Records into the Clean Collection

The shared collection contains 593 records from four companies.

We will copy only Lockheed Martin and Warner Bros. Discovery records into the clean collection.

The existing documents, metadata, and embeddings will be reused. No new embeddings are generated.

Expected result:

149 Lockheed Martin + 173 Warner Bros. Discovery = 322 chunks.

In [ ]:
source_data = collection.get(
    include=["documents", "metadatas", "embeddings"]
)

lmt_wbd_records = []

for document, metadata, embedding in zip(
    source_data["documents"],
    source_data["metadatas"],
    source_data["embeddings"]
):
    if metadata["company"] in [
        "Lockheed Martin",
        "Warner Bros. Discovery"
    ]:
        lmt_wbd_records.append({
            "id": metadata["chunk_id"],
            "document": document,
            "metadata": metadata,
            "embedding": embedding
        })

print("Records selected:", len(lmt_wbd_records))

## 33.2 Verify the Selected Records

Before inserting the records into the clean collection, we verify the company distribution.

Expected:

- Lockheed Martin: 149
- Warner Bros. Discovery: 173
- Total: 322

In [ ]:
selected_company_counts = {}

for record in lmt_wbd_records:
    company = record["metadata"]["company"]
    selected_company_counts[company] = (
        selected_company_counts.get(company, 0) + 1
    )

print("Selected records by company:")

for company, count in sorted(selected_company_counts.items()):
    print(f"  {company}: {count}")

print("\nTotal:", len(lmt_wbd_records))

In [ ]:
#Insert the clean records
clean_collection.add(
    ids=[record["id"] for record in lmt_wbd_records],
    documents=[record["document"] for record in lmt_wbd_records],
    embeddings=[record["embedding"] for record in lmt_wbd_records],
    metadatas=[record["metadata"] for record in lmt_wbd_records]
)

print("Clean collection count:", clean_collection.count())

## 33.3 Use the Clean LMT + WBD Collection for Retrieval

Our original retrieval function uses the shared ChromaDB collection.

For the baseline RAG experiments, we will now use the clean collection containing only Lockheed Martin and Warner Bros. Discovery.

This prevents documents from other companies from affecting retrieval results.

In [ ]:
def retrieve_chunks(query, n_results=3, section=None):
    query_embedding = embedding_model.encode(query).tolist()

    if section is not None:
        results = clean_collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where={"section": section}
        )
    else:
        results = clean_collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )

    retrieved_chunks = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved_chunks.append({
            "text": document,
            "metadata": metadata,
            "distance": distance
        })

    return retrieved_chunks

print("Retrieval now uses:", clean_collection.name)

# 34. Re-test Retrieval on the Clean Collection

We previously found that some relevant chunks were retrieved but ranked below unrelated Financial Statements chunks.

We will repeat the same top-10 diagnostic using only the Lockheed Martin and Warner Bros. Discovery documents.

This gives us a clean baseline for evaluating the retrieval strategy.

In [ ]:
diagnostic_questions = [
    (
        "What risks does Lockheed Martin face from relying on U.S. government contracts?",
        "Risk Factors"
    ),
    (
        "What factors affected Lockheed Martin's financial performance?",
        "MD&A"
    ),
    (
        "What caused changes in Lockheed Martin's cash flows?",
        "MD&A"
    )
]

for question, expected_section in diagnostic_questions:

    results = retrieve_chunks(
        question,
        n_results=10
    )

    print("\n" + "=" * 80)
    print("Question:", question)
    print("Expected section:", expected_section)
    print("=" * 80)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. "
            f"{result['metadata']['chunk_id']} | "
            f"{result['metadata']['section']} | "
            f"distance={result['distance']:.4f}"
        )

## 34.1 Inspect Candidate Similarity for Re-ranking

The top-10 retrieval results contain relevant chunks, but some relevant chunks are ranked too low.

We will calculate the cosine similarity between the question and each retrieved chunk.

This is a diagnostic step before introducing a separate re-ranking model.

We will compare the original Chroma distance with the independently calculated cosine similarity.

In [ ]:
def inspect_candidate_similarity(question, n_results=10):

    results = retrieve_chunks(
        question,
        n_results=n_results
    )

    question_embedding = embedding_model.encode(
        [question]
    )

    candidate_embeddings = embedding_model.encode(
        [result["text"] for result in results]
    )

    similarities = cosine_similarity(
        question_embedding,
        candidate_embeddings
    )[0]

    ranked_candidates = []

    for result, similarity in zip(results, similarities):
        ranked_candidates.append({
            "chunk_id": result["metadata"]["chunk_id"],
            "section": result["metadata"]["section"],
            "chroma_distance": result["distance"],
            "cosine_similarity": similarity
        })

    return ranked_candidates

In [ ]:
question = (
    "What risks does Lockheed Martin face "
    "from relying on U.S. government contracts?"
)

candidates = inspect_candidate_similarity(
    question,
    n_results=10
)

for rank, candidate in enumerate(candidates, start=1):
    print(
        f"{rank}. "
        f"{candidate['chunk_id']} | "
        f"{candidate['section']} | "
        f"Chroma distance={candidate['chroma_distance']:.4f} | "
        f"Cosine similarity={candidate['cosine_similarity']:.4f}"
    )

## 34.2 Inspect Competing Retrieved Chunks

The embedding model ranked two Financial Statements chunks above the Risk Factors chunks.

We will inspect the highest-ranked Financial Statements chunk and the highest-ranked Risk Factors chunk.

This helps us understand why the embedding model considers them semantically similar to the question.

In [ ]:
question = (
    "What risks does Lockheed Martin face "
    "from relying on U.S. government contracts?"
)

candidates = inspect_candidate_similarity(
    question,
    n_results=10
)

for candidate in candidates[:3]:

    result = retrieve_chunks(
        question,
        n_results=10
    )

    matching_chunk = next(
        item for item in result
        if item["metadata"]["chunk_id"] == candidate["chunk_id"]
    )

    print("\n" + "=" * 80)
    print("Chunk ID:", candidate["chunk_id"])
    print("Section:", candidate["section"])
    print("Chroma distance:", candidate["chroma_distance"])
    print("Cosine similarity:", candidate["cosine_similarity"])
    print("=" * 80)
    print(matching_chunk["text"][:2500])

## 34.3 Test Retrieval with Different Question Types

The previous retrieval test focused heavily on U.S. Government contracts.

In this experiment, we test several different questions covering business,
employees, suppliers, cybersecurity, financial performance, cash flow,
assets, and earnings.

The purpose is to observe whether semantic retrieval finds useful evidence
when the wording and topic vary.

No section metadata filter is used in this test.

In [ ]:
test_questions = [
    "What are Lockheed Martin's main business activities?",
    "How many employees does Lockheed Martin have?",
    "What problems could Lockheed Martin face if its suppliers fail to perform?",
    "What cybersecurity risks does Lockheed Martin face?",
    "Why did Lockheed Martin's financial results change in 2025?",
    "What affected Lockheed Martin's cash flow in 2025?",
    "How much did Lockheed Martin report in total assets?",
    "How much net income did Lockheed Martin earn in 2025?"
]

for question in test_questions:

    results = retrieve_chunks(
        question,
        n_results=5
    )

    print("\n" + "=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. "
            f"{result['metadata']['chunk_id']} | "
            f"{result['metadata']['section']} | "
            f"Distance: {result['distance']:.4f}"
        )

## 34.5 Inspect Frequently Retrieved Financial Statement Chunk

The retrieval tests repeatedly ranked `LMT_Financial_Statements_004`
near the top for several different questions.

Before changing the retrieval pipeline, we inspect this chunk directly.

The goal is to determine whether the chunk genuinely contains useful
information for multiple questions or whether the embedding model is
over-ranking a broad financial chunk.

In [ ]:
chunk_to_inspect = clean_collection.get(
    ids=["LMT_Financial_Statements_004"],
    include=["documents", "metadatas"]
)

print("Chunk ID:", chunk_to_inspect["ids"][0])
print("Section:", chunk_to_inspect["metadatas"][0]["section"])
print("=" * 100)
print(chunk_to_inspect["documents"][0])

In [ ]:
wbd_test_questions = [
    "What are Warner Bros. Discovery's main business activities?",
    "How many employees does Warner Bros. Discovery have?",
    "What major business risks does Warner Bros. Discovery face?",
    "What cybersecurity risks does Warner Bros. Discovery face?",
    "Why did Warner Bros. Discovery's financial results change in 2025?",
    "What affected Warner Bros. Discovery's cash flow in 2025?",
    "How much did Warner Bros. Discovery report in total assets?",
    "How much net income did Warner Bros. Discovery earn in 2025?"
]

for question in wbd_test_questions:
    results = retrieve_chunks(question, n_results=5)

    print("\n" + "=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. "
            f"{result['metadata']['chunk_id']} | "
            f"{result['metadata']['section']} | "
            f"Distance: {result['distance']:.4f}"
        )

## 34.6 Inspect Frequently Retrieved WBD Financial Statement Chunks

Several Warner Bros. Discovery retrieval tests repeatedly returned the
same Financial Statements chunks:

- WBD_Financial_Statements_010
- WBD_Financial_Statements_006
- WBD_Financial_Statements_047

These chunks ranked highly even for questions about business activities,
employees, business risks, and cybersecurity.

We inspect their contents to determine whether they genuinely contain
broad company information or whether the embedding model is over-ranking
them.

In [ ]:
wbd_chunks_to_inspect = [
    "WBD_Financial_Statements_010",
    "WBD_Financial_Statements_006",
    "WBD_Financial_Statements_047"
]

chunk_data = clean_collection.get(
    ids=wbd_chunks_to_inspect,
    include=["documents", "metadatas"]
)

for chunk_id, document, metadata in zip(
    chunk_data["ids"],
    chunk_data["documents"],
    chunk_data["metadatas"]
):
    print("\n" + "=" * 100)
    print("Chunk ID:", chunk_id)
    print("Section:", metadata["section"])
    print("=" * 100)
    print(document[:3000])

## 34.7 Validate Warner Bros. Discovery Risk Factors

The previous retrieval tests did not consistently return Warner Bros.
Discovery Risk Factors for risk-related questions.

Before changing retrieval, we verify that the Risk Factors chunks were
correctly extracted and contain relevant risk information.

This separates a document-processing problem from a retrieval problem.

In [ ]:
wbd_risk_check = clean_collection.get(
    ids=["WBD_Risk_Factors_001"],
    include=["documents", "metadatas"]
)

print("Chunk ID:", wbd_risk_check["ids"][0])
print("Section:", wbd_risk_check["metadatas"][0]["section"])
print("=" * 100)
print(wbd_risk_check["documents"][0][:3000])

## 34.8 Test Company-Filtered Retrieval

When a user explicitly asks about a company, the company identity is known.

Therefore, we can safely restrict retrieval to chunks belonging to that
company while still allowing all document sections to compete.

This experiment compares retrieval before and after applying the company
metadata filter.

The section is intentionally NOT filtered because useful evidence may
appear in Financial Statements, Notes, MD&A, Business, or Risk Factors.

In [ ]:
def retrieve_chunks_by_company(
    query,
    company,
    n_results=5
):
    query_embedding = embedding_model.encode(query).tolist()

    results = clean_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where={"company": company}
    )

    retrieved_chunks = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved_chunks.append({
            "text": document,
            "metadata": metadata,
            "distance": distance
        })

    return retrieved_chunks

In [ ]:
wbd_filtered_results = {}

for question in wbd_test_questions:

    results = retrieve_chunks_by_company(
        query=question,
        company="Warner Bros. Discovery",
        n_results=5
    )

    wbd_filtered_results[question] = results

    print("\n" + "=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    for rank, result in enumerate(results, start=1):
        print(
            f"{rank}. "
            f"{result['metadata']['chunk_id']} | "
            f"{result['metadata']['section']} | "
            f"Distance: {result['distance']:.4f}"
        )

# 35. RAG Retrieval Benchmark

We will inspect a small set of retrieved chunks and manually label whether each chunk contains useful evidence for the question.

The goal is not to check whether the chunk comes from the expected 10-K section. Instead, we check whether the chunk actually contains evidence that can help answer the question.

We will start with three WBD questions:
1. Main business activities
2. Cybersecurity risks
3. Financial results in 2025

In [ ]:
benchmark_questions = [
    "What are Warner Bros. Discovery's main business activities?",
    "What cybersecurity risks does Warner Bros. Discovery face?",
    "Why did Warner Bros. Discovery's financial results change in 2025?"
]

for question in benchmark_questions:
    results = retrieve_chunks_by_company(
        query=question,
        company="Warner Bros. Discovery",
        n_results=5
    )

    print("\n" + "=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    for rank, result in enumerate(results, start=1):

        print(f"\n--- Rank {rank} ---")
        print("Chunk ID:", result["metadata"]["chunk_id"])
        print("Section:", result["metadata"]["section"])
        print("Distance:", round(result["distance"], 4))
        print("\nText:")
        print(result["text"][:1500])

## 35.1 RAG Retrieval Depth Check

For the cybersecurity question, retrieve more candidates from the WBD collection.

The goal is to determine whether the relevant cybersecurity risk chunk is present deeper in the retrieval results.

If it is present, reranking may improve the ordering.

If it is not present, the problem is with the initial embedding retrieval rather than the ranking stage.

In [ ]:
cybersecurity_question = "What cybersecurity risks does Warner Bros. Discovery face?"

cybersecurity_results = retrieve_chunks_by_company(
    query=cybersecurity_question,
    company="Warner Bros. Discovery",
    n_results=20
)

for rank, result in enumerate(cybersecurity_results, start=1):

    print("\n" + "=" * 100)
    print(f"RANK {rank}")
    print("Chunk ID:", result["metadata"]["chunk_id"])
    print("Section:", result["metadata"]["section"])
    print("Distance:", round(result["distance"], 4))
    print("\nText:")
    print(result["text"][:800])

---
---

# 36 Import the UPS and Uber source data

> We will not modify the existing **LMT/WBD** data or **ChromaDB collection**.

In [ ]:
UPS_PATH = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/UPS/10-K_2026-02-17_0001628280-26-008432.txt"

UBER_PATH = "/content/drive/MyDrive/data_handling_and_machine_learning/RAG/UBER/10-K_2026-02-13_0001543151-26-000015.txt"

## 36.1 Inspect the UPS and Uber source document

First, we inspect the beginning and end of the UPS and Uber document to understand its format before extracting the four RAG sections.

```
We will look for:
- SEC/XBRL metadata
- table of contents
- section headings
- HTML/text artifacts
- the actual 10-K content
```


In [ ]:
with open(UPS_PATH, "r", encoding="utf-8") as f:
    ups_text = f.read()

with open(UBER_PATH, "r", encoding="utf-8") as f:
    uber_text = f.read()

print("UPS total characters:", len(ups_text))
print("Uber total characters:", len(uber_text))

print("\n--- BEGINNING ---")
print(ups_text[:3000])

print("\n--- END ---")
print(ups_text[-3000:])

print("\n--- BEGINNING ---")
print(uber_text[:3000])

print("\n--- END ---")
print(uber_text[-3000:])

## 36.2 Clean the UPS source text

The raw SEC/XBRL extraction contains metadata and formatting artifacts.
Before locating the 10-K sections, we normalize non-breaking spaces so that
section-heading searches are more reliable.

We keep the original `ups_text` unchanged and create a separate cleaned copy.

In [ ]:
ups_clean = ups_text.replace("\xa0", " ")

print("Original characters:", len(ups_text))
print("Cleaned characters:", len(ups_clean))

## 36.3 Locate UPS 10-K section headings

The raw document may contain section headings in the table of contents as well
as in the actual 10-K body.

We therefore find all occurrences first instead of assuming that the first
match is the actual section.

The four sections we need are:

1. Business
2. Risk Factors
3. MD&A
4. Financial Statements

In [ ]:
section_patterns = {
    "Business": "ITEM 1. BUSINESS",
    "Risk Factors": "ITEM 1A. RISK FACTORS",
    "MD&A": "ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS",
    "Financial Statements": "ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA"
}

for section_name, pattern in section_patterns.items():
    positions = []
    start = 0

    while True:
        position = ups_clean.find(pattern, start)

        if position == -1:
            break

        positions.append(position)
        start = position + 1

    print(f"\n{section_name}")
    print("Pattern:", pattern)
    print("Occurrences:", positions)

## 36.4 Locate the UPS section headings

The extracted UPS document places the item number and section title on
separate lines. We therefore search for the headings using the same
line structure as the source document.

In [ ]:
import re

section_patterns = {
    "Business": r"ITEM\s*1\.\s*BUSINESS",
    "Risk Factors": r"ITEM\s*1A\.\s*RISK FACTORS",
    "MD&A": r"ITEM\s*7\.\s*MANAGEMENT['’]S\s+DISCUSSION\s+AND\s+ANALYSIS",
    "Financial Statements": r"ITEM\s*8\.\s*FINANCIAL STATEMENTS\s+AND\s+SUPPLEMENTARY DATA"
}

for section_name, pattern in section_patterns.items():
    matches = list(re.finditer(pattern, ups_clean, flags=re.IGNORECASE))

    print(f"\n{section_name}")
    print("Occurrences:", [match.start() for match in matches])

## 36.5 Verify the UPS section boundaries

The first occurrences of the section headings appear close together and are
likely from the table of contents.

We therefore inspect the text around the later candidate positions before
using them as extraction boundaries.

In [ ]:
candidate_positions = {
    "Business": 109929,
    "Risk Factors": 141382,
    "MD&A": 195027,
    "Financial Statements": 304363
}

for section_name, position in candidate_positions.items():
    print(f"\n{'=' * 70}")
    print(section_name, "at position", position)
    print("=" * 70)
    print(ups_clean[position:position + 1000])

## 36.6 Find the UPS section end boundaries

We have verified the actual starts of the four sections.

Now we locate the next major Item headings so that each section can be
extracted without accidentally including the following section.

In [ ]:
boundary_patterns = {
    "Item 1B": r"Item\s*1B\.",
    "Item 7A": r"Item\s*7A\.",
    "Item 9": r"Item\s*9\."
}

for boundary_name, pattern in boundary_patterns.items():
    matches = list(re.finditer(pattern, ups_clean, flags=re.IGNORECASE))

    print(f"\n{boundary_name}")
    print("Occurrences:", [match.start() for match in matches])

# 37 Extract the four UPS 10-K sections

Using the verified section boundaries, extract:

- Business
- Risk Factors
- MD&A
- Financial Statements

The extracted text will be used as the source material for RAG chunking.

In [ ]:
ups_sections = {
    "Business": ups_clean[109929:141382],
    "Risk Factors": ups_clean[141382:183263],
    "MD&A": ups_clean[195027:298013],
    "Financial Statements": ups_clean[304363:510117]
}

for section_name, text in ups_sections.items():
    print(f"{section_name}: {len(text):,} characters")

## 37.1 Validate the extracted UPS sections

Before creating RAG chunks, check the beginning and end of each extracted
section.

This helps confirm that our manually verified boundaries produced clean
sections.

In [ ]:
for section_name, text in ups_sections.items():
    print(f"\n{'=' * 70}")
    print(section_name)
    print("=" * 70)

    print("\n--- START ---")
    print(text[:500])

    print("\n--- END ---")
    print(text[-500:])

## 37.2 Chunk the UPS sections

We split each extracted section into overlapping chunks using the same
methodology used for the LMT/WBD RAG experiment:

- Chunk size: 3,000 characters
- Chunk overlap: 300 characters
- RecursiveCharacterTextSplitter

The overlap helps preserve context when an important sentence or paragraph
falls across a chunk boundary.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300
)

ups_chunks = {}

for section_name, text in ups_sections.items():
    ups_chunks[section_name] = text_splitter.split_text(text)

for section_name, chunks in ups_chunks.items():
    print(f"{section_name}: {len(chunks)} chunks")

print("\nTotal chunks:", sum(len(chunks) for chunks in ups_chunks.values()))

### 37.2.1 Inspect sample UPS chunks

Before adding metadata, inspect a few chunks from different sections.

We want to confirm that the chunks contain coherent text and that the
overlap/splitting process did not create unexpected fragments.

In [ ]:
samples = [
    ("Business", 0),
    ("Risk Factors", 0),
    ("MD&A", 0),
    ("Financial Statements", 0)
]

for section_name, chunk_index in samples:
    chunk = ups_chunks[section_name][chunk_index]

    print(f"\n{'=' * 70}")
    print(f"{section_name} — Chunk {chunk_index + 1}")
    print("=" * 70)
    print(chunk[:1000])

## 37.3 Create standardized UPS RAG records

Each chunk receives metadata describing:

- company
- ticker
- filing type
- filing year
- section
- chunk ID
- source

This gives the retrieval system enough information to identify where each
piece of evidence came from.

In [ ]:
company_metadata = {
    "company": "United Parcel Service",
    "ticker": "UPS",
    "filing_type": "10-K",
    "filing_year": 2025,
    "source": "SEC EDGAR"
}

ups_records = []

for section_name, chunks in ups_chunks.items():
    for index, chunk in enumerate(chunks, start=1):

        record = {
            "text": chunk,
            "metadata": {
                **company_metadata,
                "section": section_name,
                "chunk_id": f"UPS_{section_name.replace(' ', '_')}_{index:03d}"
            }
        }

        ups_records.append(record)

print("Total records:", len(ups_records))
print("\nFirst record:")
print(ups_records[0])

# 38 Prepare the Uber 10-K

We apply the same preprocessing methodology used for UPS.

First, normalize the non-breaking spaces in the raw SEC text.

In [ ]:
uber_clean = uber_text.replace("\xa0", " ")

print("Original characters:", len(uber_text))
print("Cleaned characters:", len(uber_clean))

## 38.1 Locate the Uber 10-K section headings

First, search for the four logical sections.

Because SEC filings often contain the headings multiple times, we will collect
all occurrences first rather than assuming the first match is the actual section.

In [ ]:
section_patterns = {
    "Business": r"ITEM\s*1\.\s*BUSINESS",
    "Risk Factors": r"ITEM\s*1A\.\s*RISK FACTORS",
    "MD&A": r"ITEM\s*7\.\s*MANAGEMENT['’]S\s+DISCUSSION\s+AND\s+ANALYSIS",
    "Financial Statements": r"ITEM\s*8\.\s*FINANCIAL STATEMENTS\s+AND\s+SUPPLEMENTARY\s+DATA"
}

for section_name, pattern in section_patterns.items():
    matches = list(re.finditer(pattern, uber_clean, flags=re.IGNORECASE))

    print(f"\n{section_name}")
    print("Occurrences:", [match.start() for match in matches])

## 38.2 Verify the Uber section boundaries

The table of contents appears around character 44,000.

We now inspect the later candidate headings to identify the actual
10-K section starts before extracting any text.

In [ ]:
candidate_positions = {
    "Business": [52456],
    "Risk Factors": [83362],
    "MD&A": [328871, 329403],
    "Financial Statements": [414324]
}

for section_name, positions in candidate_positions.items():

    for position in positions:
        print(f"\n{'=' * 70}")
        print(f"{section_name} at position {position}")
        print("=" * 70)
        print(uber_clean[position:position + 1000])

## 38.3 Find the Uber section end boundaries

The actual section starts have been verified.

Now locate the next major Item headings so we can extract each section
without including the following section.

In [ ]:
boundary_patterns = {
    "Item 1B": r"Item\s*1B\.",
    "Item 7A": r"Item\s*7A\.",
    "Item 9": r"Item\s*9\."
}

for boundary_name, pattern in boundary_patterns.items():
    matches = list(re.finditer(pattern, uber_clean, flags=re.IGNORECASE))

    print(f"\n{boundary_name}")
    print("Occurrences:", [match.start() for match in matches])

## 38.4 Extract the four Uber 10-K sections

Using the verified section boundaries, extract the same four logical
sections used for UPS and the earlier LMT/WBD experiment.

In [ ]:
uber_sections = {
    "Business": uber_clean[52456:83362],
    "Risk Factors": uber_clean[83362:315519],
    "MD&A": uber_clean[328871:408177],
    "Financial Statements": uber_clean[414324:626870]
}

for section_name, text in uber_sections.items():
    print(f"{section_name}: {len(text):,} characters")

print("\nTotal extracted characters:",
      sum(len(text) for text in uber_sections.values()))

### 38.4.1 Validate the extracted Uber sections

Check the beginning and end of each extracted section to confirm that the
verified boundaries produced clean section extraction.

In [ ]:
for section_name, text in uber_sections.items():
    print(f"\n{'=' * 70}")
    print(section_name)
    print("=" * 70)

    print("\n--- START ---")
    print(text[:500])

    print("\n--- END ---")
    print(text[-500:])

## 38.5 Chunk the Uber sections

Split each extracted Uber section into overlapping chunks using the same
3,000-character chunk size and 300-character overlap used for UPS.

In [ ]:
uber_chunks = {}

for section_name, text in uber_sections.items():
    uber_chunks[section_name] = text_splitter.split_text(text)

for section_name, chunks in uber_chunks.items():
    print(f"{section_name}: {len(chunks)} chunks")

print("\nTotal chunks:", sum(len(chunks) for chunks in uber_chunks.values()))

### 38.5.1 Inspect sample Uber chunks

Inspect the first chunk from each section to confirm that the same chunking
methodology produces coherent text across the Uber filing.

In [ ]:
samples = [
    ("Business", 0),
    ("Risk Factors", 0),
    ("MD&A", 0),
    ("Financial Statements", 0)
]

for section_name, chunk_index in samples:
    chunk = uber_chunks[section_name][chunk_index]

    print(f"\n{'=' * 70}")
    print(f"{section_name} — Chunk {chunk_index + 1}")
    print("=" * 70)
    print(chunk[:1000])

## 38.6 Create standardized Uber RAG records

Create one standardized record for every Uber chunk using the same metadata
schema used for UPS.

Keeping the schema identical allows the two companies to be combined later
without changing the retrieval pipeline.

In [ ]:
uber_metadata = {
    "company": "Uber Technologies",
    "ticker": "UBER",
    "filing_type": "10-K",
    "filing_year": 2025,
    "source": "SEC EDGAR"
}

uber_records = []

for section_name, chunks in uber_chunks.items():
    for index, chunk in enumerate(chunks, start=1):

        record = {
            "text": chunk,
            "metadata": {
                **uber_metadata,
                "section": section_name,
                "chunk_id": f"UBER_{section_name.replace(' ', '_')}_{index:03d}"
            }
        }

        uber_records.append(record)

print("Total Uber records:", len(uber_records))
print("\nFirst record:")
print(uber_records[0])

# 39 Combine UPS and Uber Records

Combine the standardized records from UPS and Uber into one dataset.

Expected:
- UPS: 147 records
- Uber: 226 records
- Combined: 373 records
- Every chunk should have a unique `chunk_id`.

In [ ]:
# Combine the standardized records
final_records = ups_records + uber_records

print("Total combined records:", len(final_records))

# Check unique chunk IDs
chunk_ids = [record["metadata"]["chunk_id"] for record in final_records]

print("Unique chunk IDs:", len(set(chunk_ids)))

# Check company distribution
from collections import Counter

company_counts = Counter(
    record["metadata"]["company"]
    for record in final_records
)

print("\nRecords by company:")
for company, count in company_counts.items():
    print(f"{company}: {count}")

## 39.1 Validate Final RAG Records

Check that every record has:
- text
- company
- ticker
- filing type
- filing year
- source
- section
- chunk ID

Also verify the number of chunks in each section for UPS and Uber.

In [ ]:
required_metadata = [
    "company",
    "ticker",
    "filing_type",
    "filing_year",
    "source",
    "section",
    "chunk_id"
]

# Check required fields
missing_fields = []

for i, record in enumerate(final_records):
    if not record.get("text"):
        missing_fields.append((i, "text"))

    for field in required_metadata:
        if field not in record["metadata"]:
            missing_fields.append((i, field))

print("Records checked:", len(final_records))
print("Missing fields:", len(missing_fields))

# Count records by company and section
from collections import Counter

section_counts = Counter(
    (
        record["metadata"]["company"],
        record["metadata"]["section"]
    )
    for record in final_records
)

print("\nSection counts:")

for (company, section), count in sorted(section_counts.items()):
    print(f"{company:25} | {section:22} | {count}")

# Check allowed sections
allowed_sections = {
    "Business",
    "Risk Factors",
    "MD&A",
    "Financial Statements"
}

unexpected_sections = {
    record["metadata"]["section"]
    for record in final_records
    if record["metadata"]["section"] not in allowed_sections
}

print("\nUnexpected sections:", unexpected_sections)

## 39.2 Test a Single Embedding

Generate an embedding for the first UPS/UBER RAG chunk.

Expected:
- One numerical vector
- 384 dimensions
- No errors

In [ ]:
test_text = final_records[0]["text"]

test_embedding = embedding_model.encode(test_text)

print("Embedding type:", type(test_embedding))
print("Embedding shape:", test_embedding.shape)
print("First 5 values:", test_embedding[:5])

# 40 Generate Embeddings for All RAG Chunks

Convert all 373 standardized RAG chunks into 384-dimensional embeddings.

We will keep the original records unchanged and store the embeddings separately.

Expected:
- 373 embeddings
- Each embedding has 384 dimensions

In [ ]:
texts = [record["text"] for record in final_records]

embeddings = embedding_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)

## 40.1 Validate Generated Embeddings

Confirm that:
- The number of embeddings matches the number of RAG records.
- Every embedding has 384 dimensions.
- There are no NaN or infinite values.

Only after this validation will we create the new UPS + Uber ChromaDB collection.

In [ ]:
import numpy as np

print("Number of records:", len(final_records))
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])

print("Contains NaN:", np.isnan(embeddings).any())
print("Contains infinity:", np.isinf(embeddings).any())

print("\nEmbedding validation complete.")

# 41 Create UPS + Uber RAG Collection

Create a separate ChromaDB collection for the final UPS + Uber RAG experiment.

The collection will contain:
- 373 text chunks
- 373 MiniLM embeddings
- Metadata for company, ticker, filing, section, source, and chunk ID

Do not modify the existing RAG collections.

In [ ]:
import chromadb

# Connect to the existing persistent ChromaDB location
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/data_handling_and_machine_learning/week_04/chroma_db")

# Create a separate collection for UPS + Uber
ups_uber_collection = chroma_client.get_or_create_collection(name="investiq_rag_ups_uber")

print("Collection:", ups_uber_collection.name)
print("Current count:", ups_uber_collection.count())

Collection: investiq_rag_ups_uber
Current count: 373


## 41.1 Insert UPS + Uber Records into ChromaDB

Insert the 373 validated RAG records into the new
`investiq_rag_ups_uber` collection.

Each ChromaDB record contains:
- The original chunk text
- Its MiniLM embedding
- Standardized metadata
- A unique chunk ID

In [ ]:
documents = [record["text"] for record in final_records]

metadatas = [record["metadata"] for record in final_records]

ids = [record["metadata"]["chunk_id"] for record in final_records]

ups_uber_collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
    ids=ids
)

print("Inserted records:", len(ids))
print("Collection count:", ups_uber_collection.count())

## 41.2 Verify UPS + Uber ChromaDB Collection

Verify that the new collection contains:
- 373 records
- 147 UPS records
- 226 Uber records
- The expected four sections
- Unique chunk IDs

This confirms that the ChromaDB insertion preserved the dataset structure.

In [ ]:
# Retrieve metadata from the collection
stored_data = ups_uber_collection.get(include=["metadatas"])

stored_metadatas = stored_data["metadatas"]

print("Total records:", len(stored_metadatas))

# Count companies
company_counts = Counter(metadata["company"] for metadata in stored_metadatas)

print("\nCompanies:")
for company, count in sorted(company_counts.items()):
    print(f"{company}: {count}")

# Count sections
section_counts = Counter(metadata["section"] for metadata in stored_metadatas)

print("\nSections:")
for section, count in sorted(section_counts.items()):
    print(f"{section}: {count}")

# Check unique IDs
stored_ids = stored_data["ids"]

print("\nUnique IDs:", len(set(stored_ids)))

# 42 UPS + Uber Retrieval Function

Create a retrieval function that:

1. Converts the user's question into a MiniLM embedding.
2. Searches the UPS + Uber ChromaDB collection.
3. Returns the top matching chunks.
4. Includes the chunk text, metadata, and Chroma distance.

We will initially search across all four sections.

In [ ]:
def retrieve_ups_uber(query, n_results=5, company=None):
    query_embedding = embedding_model.encode(query).tolist()

    if company is not None:
        results = ups_uber_collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where={"company": company}
        )
    else:
        results = ups_uber_collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results
        )

    retrieved_chunks = []

    for document, metadata, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved_chunks.append({
            "text": document,
            "metadata": metadata,
            "distance": distance
        })

    return retrieved_chunks

print("Retrieval function ready.")

Retrieval function ready.


## 42.1 Retrieval Test 1 — UPS Government Contract Risks

Question:

"What are the major risks UPS faces from its dependence on the U.S. government?"

Retrieve the top 5 chunks from UPS only.

Evaluation focus:
- Does the retrieved text actually contain evidence relevant to the question?
- Which sections are retrieved?
- Are relevant Risk Factors chunks ranked highly?
- Are unrelated chunks ranked above relevant evidence?

Do not generate an answer with Qwen yet.

In [ ]:
question = "What are the major risks UPS faces from its dependence on the U.S. government?"

retrieved = retrieve_ups_uber(
    question,
    n_results=5,
    company="United Parcel Service"
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Company: {result['metadata']['company']}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

## 42.2 Retrieval Test 2 — UPS Business Structure

Question:

"What are UPS's main business segments?"

Retrieve the top 5 UPS chunks.

Evaluation focus:
- Does the Business section appear near the top?
- Does the retrieved text directly describe UPS's business segments?
- Is the evidence more focused than in the previous test?

Do not generate an answer with Qwen yet.

In [ ]:
question = "What are UPS's main business segments?"

retrieved = retrieve_ups_uber(
    question,
    n_results=5,
    company="United Parcel Service"
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

## 42.3 Retrieval Test 3 — UPS Financial Information

Question:

"What was UPS's total revenue in 2025?"

Retrieve the top 5 UPS chunks.

Evaluation focus:
- Is the Financial Statements section retrieved?
- Does a retrieved chunk contain the exact revenue figure?
- Is the relevant evidence ranked highly?

Do not use Qwen yet.

In [ ]:
question = "What was UPS's total revenue in 2025?"

retrieved = retrieve_ups_uber(
    question,
    n_results=5,
    company="United Parcel Service"
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

## 42.4 Retrieval Test 4 — Uber Financial Information

Question:

"What was Uber's total revenue in 2025?"

Retrieve the top 5 Uber chunks.

Evaluation focus:
- Does the retriever find the correct financial evidence?
- Which section contains the answer?
- Is the answer present in the top results?
- How does retrieval behavior compare with UPS?

In [ ]:
question = "What was Uber's total revenue in 2025?"

retrieved = retrieve_ups_uber(
    question,
    n_results=5,
    company="Uber Technologies"
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

## 42.5 Retrieval Test 5 — Uber Business Information

Question:
"What are Uber's main business segments?"

Retrieve the top 5 Uber chunks.

Evaluation focus:
- Does the retriever find business-related evidence?
- Are the relevant chunks near the top?
- Is the answer supported by the retrieved text?
- Which sections are retrieved?

In [ ]:
question = "What are Uber's main business segments?"

retrieved = retrieve_ups_uber(
    question,
    n_results=5,
    company="Uber Technologies"
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

## 42.6 Retrieval Test 6 — Uber Risk Information

Question:
"What cybersecurity risks does Uber identify in its 2025 10-K?"

Retrieve the top 5 Uber chunks.

Evaluation focus:
- Does retrieval find Risk Factors evidence?
- How high does the relevant risk chunk rank?
- Is the information actually sufficient to answer the question?
- Are irrelevant Financial Statements or MD&A chunks ranked above it?

In [ ]:
question = "What cybersecurity risks does Uber identify in its 2025 10-K?"

retrieved = retrieve_ups_uber(
    question,
    n_results=5,
    company="Uber Technologies"
)

for i, result in enumerate(retrieved, start=1):
    print("=" * 80)
    print(f"Rank {i}")
    print(f"Distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:1000])
    print()

# 43 Retrieval Baseline Summary

We summarize the retrieval behavior observed in our initial experiments.

The goal is not to measure model accuracy yet, but to record whether
relevant evidence appeared in the top 5 retrieved chunks.

In [ ]:
baseline_results = [
    {
        "company": "UPS",
        "question_type": "Business",
        "relevant_in_top_5": True,
        "relevant_rank": "Top results"
    },
    {
        "company": "UPS",
        "question_type": "Revenue",
        "relevant_in_top_5": True,
        "relevant_rank": 1
    },
    {
        "company": "UPS",
        "question_type": "Government risk",
        "relevant_in_top_5": False,
        "relevant_rank": "Not confirmed"
    },
    {
        "company": "Uber",
        "question_type": "Revenue",
        "relevant_in_top_5": True,
        "relevant_rank": 1
    },
    {
        "company": "Uber",
        "question_type": "Business",
        "relevant_in_top_5": True,
        "relevant_rank": 2
    },
    {
        "company": "Uber",
        "question_type": "Cybersecurity risk",
        "relevant_in_top_5": True,
        "relevant_rank": 3
    }
]

for result in baseline_results:
    print(
        f"{result['company']:5} | "
        f"{result['question_type']:20} | "
        f"Top 5: {result['relevant_in_top_5']} | "
        f"Rank: {result['relevant_rank']}"
    )

---
# 44 Reranker Experiment — Candidate Retrieval

Before applying a reranker, retrieve a larger candidate pool.

MiniLM will retrieve the top 10 potentially relevant chunks.
The reranker will later reorder these candidates.

We keep the company filter because the company is already known.

In [ ]:
question = "What cybersecurity risks does Uber identify in its 2025 10-K?"

candidates = retrieve_ups_uber(
    question,
    n_results=10,
    company="Uber Technologies"
)

print("Number of candidates:", len(candidates))

for i, result in enumerate(candidates, start=1):
    print(
        f"Rank {i:2} | "
        f"Distance: {result['distance']:.4f} | "
        f"Section: {result['metadata']['section']} | "
        f"Chunk: {result['metadata']['chunk_id']}"
    )

## 44.1 Reranker Model Setup

We use a cross-encoder to score each retrieved candidate against
the user's question.

Unlike embedding retrieval, the cross-encoder receives the
question and candidate text together.

The reranker is used only after MiniLM retrieval.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded.


## 44.2 Reranking the Candidate Chunks

The cross-encoder scores each candidate using both the question
and the retrieved chunk text.

Higher scores indicate stronger relevance according to the
reranker.

We will compare the reranked order with the original MiniLM order.

In [ ]:
question = "What cybersecurity risks does Uber identify in its 2025 10-K?"

candidate_pairs = [
    (question, result["text"])
    for result in candidates
]

reranker_scores = reranker.predict(candidate_pairs)

reranked = []

for result, score in zip(candidates, reranker_scores):
    reranked.append({
        "text": result["text"],
        "metadata": result["metadata"],
        "distance": result["distance"],
        "reranker_score": float(score)
    })

reranked = sorted(
    reranked,
    key=lambda x: x["reranker_score"],
    reverse=True
)

for i, result in enumerate(reranked, start=1):
    print("=" * 80)
    print(f"Reranked position: {i}")
    print(f"Reranker score: {result['reranker_score']:.4f}")
    print(f"Original MiniLM distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:500])
    print()

## 44.3 Reranked Retrieval Function

Create a reusable retrieval pipeline that combines:

1. MiniLM semantic retrieval
2. Cross-encoder reranking

The initial retrieval stage produces a candidate pool.
The reranker then reorders those candidates by query-document relevance.

In [ ]:
def retrieve_with_reranker(
    query,
    n_candidates=10,
    n_results=5,
    company=None
):
    candidates = retrieve_ups_uber(
        query,
        n_results=n_candidates,
        company=company
    )

    candidate_pairs = [
        (query, result["text"])
        for result in candidates
    ]

    scores = reranker.predict(candidate_pairs)

    reranked = []

    for result, score in zip(candidates, scores):
        reranked.append({
            "text": result["text"],
            "metadata": result["metadata"],
            "distance": result["distance"],
            "reranker_score": float(score)
        })

    reranked.sort(
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked[:n_results]

In [ ]:
print("retrieve_ups_uber:", "available" if "retrieve_ups_uber" in globals() else "NOT available")
print("retrieve_with_reranker:", "available" if "retrieve_with_reranker" in globals() else "NOT available")

# 45 MiniLM vs Reranker — Controlled Comparison

We compare the original MiniLM ranking with the
MiniLM + cross-encoder reranking pipeline.

For each question, we record the position of the
known relevant evidence chunk.

The goal is to determine whether reranking improves
the ordering of relevant evidence.

In [ ]:
evaluation_questions = [
    {
        "company": "United Parcel Service",
        "question": "What was UPS's total revenue in 2025?",
        "relevant_chunk": "UPS_Business_001"
    },
    {
        "company": "Uber Technologies",
        "question": "What was Uber's total revenue in 2025?",
        "relevant_chunk": "UBER_MD&A_003"
    },
    {
        "company": "Uber Technologies",
        "question": "What cybersecurity risks does Uber identify in its 2025 10-K?",
        "relevant_chunk": "UBER_Risk_Factors_037"
    }
]

for item in evaluation_questions:
    print("=" * 80)
    print(f"Company: {item['company']}")
    print(f"Question: {item['question']}")

    candidates = retrieve_ups_uber(
        item["question"],
        n_results=10,
        company=item["company"]
    )

    mini_rank = None

    for rank, result in enumerate(candidates, start=1):
        if result["metadata"]["chunk_id"] == item["relevant_chunk"]:
            mini_rank = rank
            break

    reranked = retrieve_with_reranker(
        item["question"],
        n_candidates=10,
        n_results=10,
        company=item["company"]
    )

    rerank_rank = None

    for rank, result in enumerate(reranked, start=1):
        if result["metadata"]["chunk_id"] == item["relevant_chunk"]:
            rerank_rank = rank
            break

    print(f"MiniLM rank: {mini_rank}")
    print(f"Reranked position: {rerank_rank}")

## 45.1 Reranker Investigation — Uber Revenue

The reranker moved the known relevant revenue chunk
from MiniLM Rank 1 to Rank 6.

We inspect the reranked candidates to understand
why the cross-encoder changed the ordering.

In [ ]:
question = "What was Uber's total revenue in 2025?"

reranked = retrieve_with_reranker(
    question,
    n_candidates=10,
    n_results=10,
    company="Uber Technologies"
)

for i, result in enumerate(reranked, start=1):
    print("=" * 80)
    print(f"Reranked position: {i}")
    print(f"Reranker score: {result['reranker_score']:.4f}")
    print(f"Original MiniLM distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:700])
    print()

## 45.2 Reranker Investigation — UPS Revenue

Inspect the reranked candidates for a question where
MiniLM already retrieved the correct evidence at Rank 1.

```
Question:
"What was UPS's total revenue in 2025?"
```

We will determine whether the reranker preserves
or replaces the relevant evidence.

---

In [ ]:
question = "What was UPS's total revenue in 2025?"

reranked = retrieve_with_reranker(
    question,
    n_candidates=10,
    n_results=10,
    company="United Parcel Service"
)

for i, result in enumerate(reranked, start=1):
    print("=" * 80)
    print(f"Reranked position: {i}")
    print(f"Reranker score: {result['reranker_score']:.4f}")
    print(f"Original MiniLM distance: {result['distance']:.4f}")
    print(f"Section: {result['metadata']['section']}")
    print(f"Chunk ID: {result['metadata']['chunk_id']}")
    print()
    print(result["text"][:700])
    print()

## 45.3 Reranker Evaluation — Direct Evidence

We evaluate retrieval based on whether the top-K results
contain direct evidence for answering the question.

A specific chunk ID is not required when multiple chunks
contain valid evidence.

Evaluation criteria:
- Direct evidence: contains the answer or directly supports it
- Supporting evidence: useful context but does not directly answer
- Irrelevant: does not help answer the question

We compare MiniLM retrieval with the reranked results.

---

# 46 Reranker Evaluation Questions

We use previously tested questions so that the comparison
is based on evidence we have already inspected.

Evaluation questions:

1. UPS — total revenue in 2025
2. Uber — total revenue in 2025
3. Uber — cybersecurity risks
4. UPS — main business segments

For each question, we compare:
- MiniLM top-5 retrieval
- Cross-encoder reranked top-5 retrieval

We evaluate whether direct evidence is present in the
top-5 results rather than requiring one specific chunk ID.

---
## 46.1 Direct Evidence Sets

For each evaluation question, we define the chunk IDs that
contain direct evidence for answering the question.

A result is considered successful if at least one of these
direct-evidence chunks appears in the top-5 retrieval results.

This allows multiple valid source chunks to support the same
question.

In [ ]:
direct_evidence = {
    "UPS_revenue": [
        "UPS_Business_001",
        "UPS_MD&A_002"
    ],

    "Uber_revenue": [
        "UBER_MD&A_003",
        "UBER_MD&A_004",
        "UBER_MD&A_013"
    ],

    "Uber_cybersecurity": [
        "UBER_Risk_Factors_037"
    ],

    "UPS_business": [
        "UPS_Business_001",
        "UPS_Business_003",
        "UPS_Business_004"
    ]
}

for question_name, chunk_ids in direct_evidence.items():
    print(question_name)
    print("Direct evidence:", chunk_ids)
    print()

## 46.2 Top-5 Direct Evidence Evaluation

The helper below checks whether at least one direct-evidence
chunk appears in the top-5 retrieved results.

We calculate the best rank of any direct-evidence chunk.

This lets us compare MiniLM and the reranker without requiring
one specific chunk to stay at the same position.

In [ ]:
def evaluate_direct_evidence(results, relevant_chunk_ids):
    retrieved_ids = [
        result["metadata"]["chunk_id"]
        for result in results
    ]

    matching_ranks = [
        rank
        for rank, chunk_id in enumerate(retrieved_ids, start=1)
        if chunk_id in relevant_chunk_ids
    ]

    return {
        "direct_evidence_found": len(matching_ranks) > 0,
        "best_rank": min(matching_ranks) if matching_ranks else None
    }

## 46.3 Reranker Evaluation — MiniLM vs Cross-Encoder

For each question we retrieve the top-5 results using:

1. MiniLM semantic retrieval
2. MiniLM candidate retrieval + cross-encoder reranking

We then compare the best rank of any direct-evidence chunk.

In [ ]:
evaluation_questions = [
    {
        "name": "UPS_revenue",
        "company": "United Parcel Service",
        "question": "What was UPS's total revenue in 2025?"
    },
    {
        "name": "Uber_revenue",
        "company": "Uber Technologies",
        "question": "What was Uber's total revenue in 2025?"
    },
    {
        "name": "Uber_cybersecurity",
        "company": "Uber Technologies",
        "question": "What cybersecurity risks does Uber identify in its 2025 10-K?"
    },
    {
        "name": "UPS_business",
        "company": "United Parcel Service",
        "question": "What are UPS's main business segments?"
    }
]

print("Evaluation questions:", len(evaluation_questions))

for item in evaluation_questions:
    print(f"- {item['name']}: {item['question']}")

## 46.4 MiniLM vs Cross-Encoder — Top-5 Evidence Comparison

For each question:

1. MiniLM retrieves 5 chunks.
2. The cross-encoder reranks the top 10 MiniLM candidates.
3. We keep the top 5 reranked chunks.
4. We compare the best rank of any direct-evidence chunk.

A lower best rank means the direct evidence appeared earlier.

In [ ]:
evaluation_results = []

for item in evaluation_questions:
    name = item["name"]
    company = item["company"]
    question = item["question"]

    relevant_ids = direct_evidence[name]

    # MiniLM top-5
    minilm_results = retrieve_ups_uber(
        question,
        n_results=5,
        company=company
    )

    # Cross-encoder reranking
    reranked_results = retrieve_with_reranker(
        question,
        n_candidates=10,
        n_results=5,
        company=company
    )

    minilm_eval = evaluate_direct_evidence(
        minilm_results,
        relevant_ids
    )

    reranked_eval = evaluate_direct_evidence(
        reranked_results,
        relevant_ids
    )

    evaluation_results.append({
        "question": name,
        "minilm_found": minilm_eval["direct_evidence_found"],
        "minilm_best_rank": minilm_eval["best_rank"],
        "reranker_found": reranked_eval["direct_evidence_found"],
        "reranker_best_rank": reranked_eval["best_rank"]
    })

for result in evaluation_results:
    print("=" * 70)
    print(result["question"])
    print(f"MiniLM:    found={result['minilm_found']}, "
          f"best_rank={result['minilm_best_rank']}")
    print(f"Reranker:  found={result['reranker_found']}, "
          f"best_rank={result['reranker_best_rank']}")

---
### RAG Retrieval and Reranking Evaluation

I evaluated the RAG pipeline using questions covering factual
retrieval, business information, and cybersecurity risks.

The baseline retriever used MiniLM semantic embeddings. A
cross-encoder reranker was then tested on the retrieved candidate
chunks.

The evaluation focused on whether direct supporting evidence was
present in the top five results and how its ranking changed after
reranking.

The results were:

| Company | Question | MiniLM | Reranker |
|---|---|---|---|
| UPS | Total revenue in 2025 | Direct evidence rank 1 | Direct evidence rank 1 |
| Uber | Total revenue in 2025 | Direct evidence rank 1 | Direct evidence rank 1 |
| Uber | Cybersecurity risks | Direct evidence rank 3 | Direct evidence rank 1 |
| UPS | Main business segments | Direct evidence rank 1 | Direct evidence rank 1 |

The reranker preserved direct evidence in all four test cases and
improved the ranking of the Uber cybersecurity evidence from rank 3
to rank 1.

Because this evaluation contains only four questions, these results
are treated as an initial evaluation rather than a general benchmark.

---

# 47 Reranked RAG — Context Preparation

We now test the complete RAG pipeline using the reranked
retrieval results.

For the cybersecurity question, the cross-encoder moved
the directly relevant Risk Factors chunk from Rank 3 to Rank 1.

We will pass the top 5 reranked chunks to Qwen and evaluate
whether the generated answer is grounded in that context.

In [ ]:
def build_reranked_context(retrieved_chunks):
    context_parts = []

    for rank, result in enumerate(retrieved_chunks, start=1):
        metadata = result["metadata"]

        source_id = metadata["chunk_id"]
        company = metadata["company"]
        section = metadata["section"]

        context_parts.append(
            f"""
            Source ID: {source_id}
            Company: {company}
            Section: {section}
            Retrieval Rank: {rank}

            Text:
            {result["text"]}
            """
        )

    return "\n".join(context_parts)

## 47.1 Retrieve and Rerank — Uber Cybersecurity

*Question:
"What cybersecurity risks does Uber identify in its 2025 10-K?"*

The pipeline will:

1. Retrieve 10 candidates with MiniLM.
2. Rerank those candidates with the cross-encoder.
3. Keep the top 5 results.
4. Build the context that will later be given to Qwen.

In [ ]:
question = "What cybersecurity risks does Uber identify in its 2025 10-K?"

reranked_results = retrieve_with_reranker(
    query=question,
    n_candidates=10,
    n_results=5,
    company="Uber Technologies"
)

for i, result in enumerate(reranked_results, start=1):
    print(f"\nRank {i}")
    print("Source ID:", result["metadata"].get("chunk_id"))
    print("Reranker score:", result["reranker_score"])
    print("Section:", result["metadata"].get("section"))
    print("Text:", result["text"][:500])


Rank 1
Source ID: UBER_Risk_Factors_037
Reranker score: -0.9435630440711975
Section: Risk Factors
Text: Security breaches could also expose us to liability under various laws and regulations across jurisdictions and increase the risk of litigation and governmental investigation. We have been subject to security and privacy incidents in the past and may be again in the future. For example, in September 2022, we experienced a cybersecurity incident where an attacker accessed certain internal corporate systems, tools and data. In October and November of 2016, outside actors downloaded the personal da

Rank 2
Source ID: UBER_Financial_Statements_010
Reranker score: -4.030342102050781
Section: Financial Statements
Text: 77
UBER TECHNOLOGIES, INC.
CONSOLIDATED STATEMENTS OF CASH FLOWS
(In millions)
Year Ended December 31,
2023
2024
2025
Cash flows from operating activities
Net income including non-controlling interests
$
2,156
$
9,845
$
10,093
Adjustments to reconcile net income to net cash

### Load Fine-tuned adapter for CPU runtime

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

ADAPTER_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/week_04/outputs/Finetuned_version01/InvestIQ_finetuned_adapter_updated")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float32
)

base_model.config.use_cache = True

trained_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

trained_model.eval()

print("Fine-tuned Qwen loaded successfully.")
print("Model device:", next(trained_model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['kasa_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Fine-tuned Qwen loaded successfully.
Model device: cpu
CUDA available: False


## 47.2 Reranked RAG — Grounded Answer Generation

Qwen will receive only the reranked retrieval context.

Grounding requirements:

- Use only the provided context.
- Every factual claim must be supported by the context.
- Do not use outside knowledge.
- Do not invent missing information.
- Use only source IDs that appear in the context.
- Return valid JSON.

In [ ]:
rag_prompt = f"""
      You are an investment research assistant.

      Answer the user's question using ONLY the retrieved source context below.

      Grounding rules:
      - Treat the retrieved context as the complete evidence boundary.
      - Every factual claim must be supported by the retrieved context.
      - Do not add facts from general knowledge.
      - If the context does not support a detail, leave that detail out.
      - Do not invent source IDs.
      - Only use source IDs that appear in the retrieved context.
      - If the retrieved context does not contain enough information to answer,
        say that the available retrieved context does not provide enough information.

      Return ONLY valid JSON using exactly this structure:

      {{
        "answer": "A concise answer supported by the retrieved context.",
        "sources": ["source_id_1", "source_id_2"]
      }}

      User question:
      {question}

      Retrieved context:
      {reranked_context}
      """

response = generate_qwen_response(rag_prompt, max_new_tokens=400)

print(response)

## 47.3 RAG Generation Evaluation — Source Attribution

The generated answer is evaluated separately from retrieval.

We check whether the source IDs returned by Qwen actually
belong to the retrieved context.

We also inspect whether Qwen cites all retrieved chunks or
only the chunks that genuinely support its answer.

In [ ]:
import json

parsed_response = json.loads(response)

generated_sources = parsed_response["sources"]

retrieved_source_ids = [
    result["metadata"]["chunk_id"]
    for result in reranked_chunks
]

print("Sources returned by Qwen:")
for source_id in generated_sources:
    print("-", source_id)

print()

print("Sources actually retrieved:")
for source_id in retrieved_source_ids:
    print("-", source_id)

## 47.4 Source Attribution — Expected Relevance

For the Uber cybersecurity question, we manually classify
the retrieved chunks based on whether they support the answer.

Categories:

- Direct: directly contains evidence used to answer the question
- Supporting: related evidence that can reasonably support the answer
- Irrelevant: retrieved context that does not support the answer

This allows us to evaluate source attribution separately
from source-ID correctness.

In [ ]:
source_relevance = {
    "UBER_Risk_Factors_037": "Direct",
    "UBER_Risk_Factors_026": "Supporting",
    "UBER_Financial_Statements_010": "Irrelevant",
    "UBER_Business_004": "Irrelevant",
    "UBER_Financial_Statements_001": "Irrelevant"
}

for source_id, relevance in source_relevance.items():
    print(f"{source_id}: {relevance}")

## 47.5 Source Attribution Precision

For this experiment, a cited source is considered relevant if
it is classified as either Direct or Supporting.

Source attribution precision is:

Relevant cited sources

----------------------

Total cited sources

This measures whether Qwen's cited sources actually support
the generated answer.

In [ ]:
relevant_sources = [
    source_id
    for source_id in generated_sources
    if source_relevance.get(source_id) in ["Direct", "Supporting"]
]

source_precision = (len(relevant_sources) / len(generated_sources))

print("Relevant cited sources:", len(relevant_sources))
print("Total cited sources:", len(generated_sources))
print(f"Source attribution precision: {source_precision:.2%}")

## 47.6 RAG Source Attribution Improvement

The previous RAG experiment showed that Qwen sometimes cited
retrieved chunks that were not actually used to support the answer.

In this experiment, we keep the same retrieved context and
retrieval results, but make the source-selection rule stricter.

Qwen should cite a source only when the generated answer contains
a factual claim supported by that specific source.

It should not cite a source merely because the source was retrieved.

In [ ]:
strict_rag_prompt = f"""
    You are an investment research assistant.

    Answer the user's question using ONLY the retrieved source context below.

    Grounding rules:
    - Treat the retrieved context as the complete evidence boundary.
    - Every factual claim must be supported by the retrieved context.
    - Do not add facts from general knowledge.
    - If the context does not support a detail, leave that detail out.
    - Do not invent source IDs.
    - Only use source IDs that appear in the retrieved context.

    Source citation rules:
    - Cite ONLY sources that directly support a factual claim in your answer.
    - Do NOT cite a source merely because it was retrieved.
    - Do NOT cite every retrieved source.
    - If only one source supports the answer, return only that source.
    - If multiple sources support different claims, cite only those sources.
    - Irrelevant retrieved sources must NOT be included in "sources".

    Return ONLY valid JSON using exactly this structure:

    {{
      "answer": "A concise answer supported by the retrieved context.",
      "sources": ["source_id_1", "source_id_2"]
    }}

    User question:
    {question}

    Retrieved context:
    {reranked_context}
    """

strict_response = generate_qwen_response(
    strict_rag_prompt,
    max_new_tokens=400
)

print(strict_response)

### RAG Source Attribution Evaluation

The reranked retrieval results were passed to the fine-tuned Qwen model to generate a grounded answer.

For the Uber cybersecurity question, the generated answer was
supported by the retrieved context. However, the model cited all five retrieved source IDs even though manual inspection showed that only two of them were relevant to the generated answer.

The source relevance classification was:

| Source | Relevance |
|---|---|
| UBER_Risk_Factors_037 | Direct |
| UBER_Risk_Factors_026 | Supporting |
| UBER_Financial_Statements_010 | Irrelevant |
| UBER_Business_004 | Irrelevant |
| UBER_Financial_Statements_001 | Irrelevant |

Source attribution precision was calculated as:

Relevant cited sources
----------------------
Total cited sources

2 / 5 = 40%

The result indicates that the retrieved evidence was sufficient to generate a grounded answer, but the model's source attribution was too broad. The model cited retrieved chunks that did not directly support the answer.

This suggests that source attribution should be handled more
carefully in the final RAG architecture rather than assuming that all retrieved chunks should be cited.

---
---

# 48 Current RAG Architecture

The final RAG pipeline uses a two-stage retrieval process.

1. The user's question is converted into an embedding using
   `all-MiniLM-L6-v2`.

2. ChromaDB performs semantic retrieval and returns the top candidate chunks.

3. The retrieved candidates are passed to a cross-encoder reranker (`cross-encoder/ms-marco-MiniLM-L-6-v2`).

4. The reranker scores the relevance of each candidate to the user's question and the highest-ranked evidence is selected.

5. The selected evidence is provided to the fine-tuned Qwen model.

6. Qwen generates the final InvestIQ response using the retrieved evidence as its grounding context.

```
Current architecture:

              User question
                    ↓
              Company identification / company filter
                    ↓
              MiniLM semantic retrieval
                    ↓
              Top candidate chunks
                    ↓
              Cross-encoder reranking
                    ↓
              Top relevant evidence
                    ↓
              Fine-tuned Qwen
                    ↓
              Grounded InvestIQ response

```

The evaluation showed that reranking can improve the position of relevant evidence. However, source attribution remains an area for improvement because the fine-tuned model may cite retrieved sources that do not directly support the generated answer.

# 49 Improving Source Attribution

The retrieval and reranking pipeline successfully places relevant evidence near the top of the retrieved context.

However, the first RAG generation test showed that the language model cited all retrieved chunks, including chunks that did not directly support the answer.

For the Uber cybersecurity example:
- 5 chunks were provided to the language model.
- 2 cited chunks were relevant or supporting.
- 3 cited chunks were irrelevant.
- Source attribution precision was 40%.

The next experiment tests whether providing a smaller, higher-ranked evidence set can reduce irrelevant source citations while preserving sufficient evidence to answer the question.

In [ ]:
question = "What cybersecurity risks does Uber identify in its 2025 10-K?"

top_3_results = retrieve_with_reranker(
    query=question,
    n_candidates=10,
    n_results=3,
    company="Uber Technologies"
)

for rank, result in enumerate(top_3_results, start=1):
    print(f"\n--- Rank {rank} ---")
    print("Chunk ID:", result["metadata"]["chunk_id"])
    print("Section:", result["metadata"]["section"])
    print("Reranker score:", round(result["reranker_score"], 4))
    print(result["text"][:700])

### Top-K Context Inspection

Reducing the retrieved context from five chunks to three does not fully solve the source-attribution problem.

For the Uber cybersecurity question, the reranked top three contained:

1. `UBER_Risk_Factors_037` — directly relevant cybersecurity evidence.
2. `UBER_Financial_Statements_010` — unrelated cash-flow information.
3. `UBER_Risk_Factors_026` — general technology-investment risk, but not direct cybersecurity evidence.

Therefore, simply reducing the number of retrieved chunks may remove some irrelevant context, but it does not guarantee that every supplied source supports the generated answer.

The next improvement should focus on **source attribution behavior**: the language model should cite only sources from which it actually uses factual information.

# 50 Testing Insufficient-Information Retrieval

A reliable RAG system should not only retrieve relevant information when an answer exists.

It should also recognize when the retrieved documents do not contain enough evidence to answer a question.

This test uses a question about Uber that should not be answerable from the available 2025 10-K content.

We will first inspect the retrieved and reranked evidence without asking Qwen to generate an answer.

In [ ]:
question = "What university did Uber's CEO attend?"

results = retrieve_with_reranker(
    query=question,
    n_candidates=10,
    n_results=5,
    company="Uber Technologies"
)

for rank, result in enumerate(results, start=1):
    print(f"\n--- Rank {rank} ---")
    print("Chunk ID:", result["metadata"]["chunk_id"])
    print("Section:", result["metadata"]["section"])
    print("Reranker score:", round(result["reranker_score"], 4))
    print(result["text"][:500])

# 51 Final RAG Grounding Rules

The RAG system retrieves and reranks evidence before sending it to the language model.

The language model must treat the retrieved context as the complete evidence boundary.

GROUNDING RULES:

1. Treat the retrieved context as the complete evidence boundary.
2. Use only information explicitly supported by the retrieved context.
3. Do not use outside knowledge to fill missing information.
4. If the retrieved context does not contain enough evidence to answer the question, explicitly state that the available sources do not provide enough information.
5. Cite a source ID only if you use at least one factual detail from that source in the answer.
6. A retrieved source does NOT need to be cited simply because it appears in the context.
7. If a source is irrelevant to the answer, do not cite it.
8. Never invent source IDs.
9. Every cited source ID must appear in the retrieved context.
10. Keep the answer concise and directly related to the user's question.

In [ ]:
question = "What cybersecurity risks does Uber identify in its 2025 10-K?"

reranked_results = retrieve_with_reranker(
    query=question,
    n_candidates=10,
    n_results=5,
    company="Uber Technologies"
)

reranked_context = build_reranked_context(reranked_results)

rag_prompt_v2 = f"""
      You are an investment research assistant.

      Answer the user's question using ONLY the retrieved source context below.

      GROUNDING RULES:
      1. Treat the retrieved context as the complete evidence boundary.
      2. Use only information explicitly supported by the retrieved context.
      3. Do not use outside knowledge to fill missing information.
      4. If the retrieved context does not contain enough evidence to answer the question,
        explicitly state that the available sources do not provide enough information.
      5. Cite a source ID only when the answer uses a factual detail supported by that source.
      6. Do not cite a source merely because it was included in the retrieved context.
      7. Do not cite irrelevant sources.
      8. Never invent source IDs.
      9. Every cited source ID must appear in the retrieved context.
      10. Keep the answer concise and directly related to the user's question.

      Return ONLY valid JSON using exactly this structure:

      {{
        "answer": "A concise grounded answer.",
        "sources": ["source_id_1", "source_id_2"]
      }}

      User question:
      {question}

      Retrieved context:
      {reranked_context}
      """

print(rag_prompt_v2)


      You are an investment research assistant.

      Answer the user's question using ONLY the retrieved source context below.

      GROUNDING RULES:
      1. Treat the retrieved context as the complete evidence boundary.
      2. Use only information explicitly supported by the retrieved context.
      3. Do not use outside knowledge to fill missing information.
      4. If the retrieved context does not contain enough evidence to answer the question,
        explicitly state that the available sources do not provide enough information.
      5. Cite a source ID only when the answer uses a factual detail supported by that source.
      6. Do not cite a source merely because it was included in the retrieved context.
      7. Do not cite irrelevant sources.
      8. Never invent source IDs.
      9. Every cited source ID must appear in the retrieved context.
      10. Keep the answer concise and directly related to the user's question.

      Return ONLY valid JSON using exactly thi

## 51.1 Generate a Grounded RAG Answer with Fine-Tuned Qwen

The retrieval and reranking stages are already complete.

The retrieved evidence has been placed into `rag_prompt_v2`, which defines the final grounding rules for the language model.

Now we send this prompt to the fine-tuned Qwen model running on the Tesla T4.

The model must:
- use only the retrieved context,
- follow the grounding rules,
- avoid outside knowledge,
- return only the requested JSON structure,
- and cite only source IDs that support the answer.

In [ ]:
inputs = tokenizer(
    rag_prompt_v2,
    return_tensors="pt"
).to(trained_model.device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)

In [ ]:
messages = [
    {
        "role": "user",
        "content": rag_prompt_v2
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt_text,
    return_tensors="pt"
).to(trained_model.device)

print("Input tokens:", inputs["input_ids"].shape[1])
print("Device:", inputs["input_ids"].device)

with torch.no_grad():
    outputs = trained_model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

rag_response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("\nRAG RESPONSE:")
print(rag_response)

Input tokens: 3540
Device: cpu

RAG RESPONSE:
{"answer":"The 2025 10-K identifies cybersecurity risks related to security breaches, data privacy incidents, unauthorized access, and potential legal consequences.","sources":["uber_risk_factors_037","uber_financial_statements_010"],"key_factors":["Security breaches could expose us to liability under various laws and regulations","Data privacy incidents include unauthorized access and the downloading of personal information","Unauthorized access has resulted in previous security incidents"],"risks":["Security breaches could cause reputational damage and legal exposure","Data privacy incidents may require costly remediation and regulatory responses","Unauthorized access may result in financial losses and operational disruptions"],"probability_interpretation":"","recommendation_context":"The company identified cybersecurity risks related to data privacy incidents and unauthorized access, indicating that these types of events may pose signifi

### Save RAG experiment evidence

In [ ]:
rag_experiment = {
    "question": question,
    "retrieved_information": [
        {
            "rank": rank,
            "source_id": result["metadata"]["chunk_id"],
            "section": result["metadata"]["section"],
            "company": result["metadata"]["company"],
            "reranker_score": result["reranker_score"],
            "text": result["text"]
        }
        for rank, result in enumerate(reranked_results, start=1)
    ],
    "model_output": rag_response
}

print("RAG experiment evidence saved.")
print("Question:", rag_experiment["question"])
print("Retrieved sources:", [
    item["source_id"]
    for item in rag_experiment["retrieved_information"]
])

RAG experiment evidence saved.
Question: What cybersecurity risks does Uber identify in its 2025 10-K?
Retrieved sources: ['UBER_Risk_Factors_037', 'UBER_Financial_Statements_010', 'UBER_Risk_Factors_026', 'UBER_Business_004', 'UBER_Financial_Statements_001']


In [ ]:
print("=" * 80)
print("FINAL RAG MODEL OUTPUT")
print("=" * 80)
print(rag_experiment["model_output"])

FINAL RAG MODEL OUTPUT
{"answer":"The 2025 10-K identifies cybersecurity risks related to security breaches, data privacy incidents, unauthorized access, and potential legal consequences.","sources":["uber_risk_factors_037","uber_financial_statements_010"],"key_factors":["Security breaches could expose us to liability under various laws and regulations","Data privacy incidents include unauthorized access and the downloading of personal information","Unauthorized access has resulted in previous security incidents"],"risks":["Security breaches could cause reputational damage and legal exposure","Data privacy incidents may require costly remediation and regulatory responses","Unauthorized access may result in financial losses and operational disruptions"],"probability_interpretation":"","recommendation_context":"The company identified cybersecurity risks related to data privacy incidents and unauthorized access, indicating that these types of events may pose significant threats to its bus

# Export UPS and Uber chunks to JSON

In [ ]:
import json

UPS_UBER_JSON_PATH = ("/content/drive/MyDrive/data_handling_and_machine_learning/week_05/outputs/sandip_UPS_Uber_chunks.json")

all_data = ups_uber_collection.get(include=["documents", "metadatas"])

exported_chunks = []

for document, metadata in zip(
    all_data["documents"],
    all_data["metadatas"]
):
    exported_chunks.append({
        "text": document,
        "metadata": metadata
    })

with open(UPS_UBER_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        exported_chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Export completed.")
print("Chunks exported:", len(exported_chunks))
print("Saved to:", UPS_UBER_JSON_PATH)

Export completed.
Chunks exported: 373
Saved to: /content/drive/MyDrive/data_handling_and_machine_learning/week_05/outputs/sandip_UPS_Uber_chunks.json
